In [1]:
!pip install --upgrade pip setuptools wheel -q
!pip install --upgrade cmake -q
!pip install scs --prefer-binary -q
!pip install cvxpy --prefer-binary -q


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Program Files\Python314\python.exe -m pip install --upgrade pip setuptools wheel -q

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [2]:
!pip install awswrangler -q
!pip install optbinning -q
!pip install lightgbm
!pip install xgboost
!pip install xgboost --prefer-binary
!pip install catboost


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [3]:
import awswrangler as wr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from optbinning import BinningProcess
import shutil
from warnings import simplefilter
simplefilter(action = "ignore") #, category = FutureWarning

pd.set_option('display.max_rows', 500)
from sklearn.preprocessing import LabelEncoder

(CVXPY) Jul 02 04:51:59 PM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Jul 02 04:51:59 PM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple
#import plotly.graph_objects as go
#from plotly.subplots import make_subplots
import os
#import plotly.express as px

pd.set_option('display.float_format', '{:.2f}'.format)

In [5]:
# === Conexion Athena estilo Cruce + fallback awswrangler ===
import os
import re
import sys
from pathlib import Path

import boto3
import awswrangler as wr
import pandas as pd

EXPLICIT_CREDENTIALS_SH = Path(r"c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test/credentials.sh")


def _find_dir_with_athena_client(preferred_dir: Path | None = None) -> Path | None:
    cwd = Path.cwd().resolve()
    search_roots = []

    if preferred_dir is not None:
        search_roots.append(preferred_dir)

    search_roots.extend([cwd, *cwd.parents])

    home = Path.home()
    search_roots.extend([
        home / "OneDrive - Interbank" / "conexion_aws" / "athena_conection_test",
        Path("c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test"),
    ])

    visited = set()
    for root in search_roots:
        if root in visited:
            continue
        visited.add(root)

        if not root.exists():
            continue
        if (root / "athena_client.py").exists() and (root / "athena_config.json").exists():
            return root
    return None


def _load_credentials_from_sh(sh_path: Path) -> list[str]:
    if not sh_path.exists():
        return []

    loaded_keys: list[str] = []
    pattern = re.compile(r'^\s*export\s+([A-Za-z_][A-Za-z0-9_]*)=(.*)$')

    for line in sh_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        match = pattern.match(line)
        if not match:
            continue

        key, raw_val = match.groups()
        value = raw_val.strip().strip('"').strip("'")
        if key and value:
            os.environ[key] = value
            loaded_keys.append(key)

    return loaded_keys


def _build_session(aws_region: str) -> boto3.Session:
    aws_profile = os.getenv("AWS_PROFILE")
    if aws_profile:
        return boto3.Session(profile_name=aws_profile, region_name=aws_region)
    return boto3.Session(region_name=aws_region)


def _session_is_valid(sess: boto3.Session) -> tuple[bool, str | None]:
    try:
        sts = sess.client("sts")
        _ = sts.get_caller_identity()
        return True, None
    except Exception as exc:
        return False, str(exc)


ATHENA_MODE = "wrangler"
ATHENA_DATABASE = os.getenv("ATHENA_DATABASE", "disc_comercial")
ATHENA_WORKGROUP = os.getenv("ATHENA_WORKGROUP", "primary")
ATHENA_OUTPUT = os.getenv(
    "ATHENA_OUTPUT",
    "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/athena_results/"
 )
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

client = None

credentials_file = EXPLICIT_CREDENTIALS_SH if EXPLICIT_CREDENTIALS_SH.exists() else None
if credentials_file is None:
    print(f"⚠ No se encontró credentials.sh en ruta fija: {EXPLICIT_CREDENTIALS_SH}")

preferred_dir = credentials_file.parent if credentials_file is not None else None
athena_dir = _find_dir_with_athena_client(preferred_dir=preferred_dir)
loaded_cred_keys: list[str] = []

if credentials_file is not None:
    loaded_cred_keys = _load_credentials_from_sh(credentials_file)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {credentials_file}")
elif athena_dir is not None:
    fallback_sh = athena_dir / "credentials.sh"
    loaded_cred_keys = _load_credentials_from_sh(fallback_sh)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {fallback_sh}")

try:
    session = _build_session(AWS_REGION)
except Exception:
    session = boto3.Session(region_name=AWS_REGION)

ok_session, session_error = _session_is_valid(session)
if not ok_session and session_error and "ExpiredToken" in session_error and loaded_cred_keys:
    print("⚠ Se detectó token expirado en credentials.sh. Reintentando con credenciales locales (perfil/default)...")
    for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
        os.environ.pop(key, None)
    session = _build_session(AWS_REGION)

if athena_dir is not None:
    if str(athena_dir) not in sys.path:
        sys.path.append(str(athena_dir))
    try:
        from athena_client import AthenaClient

        if credentials_file is None:
            credentials_file = athena_dir / "credentials.sh"

        client = AthenaClient(
            credentials_file=str(credentials_file),
            config_file=str(athena_dir / "athena_config.json"),
        )
        ATHENA_MODE = "athena_client"
        print(f"✓ AthenaClient cargado desde: {athena_dir}")
    except Exception as exc:
        print(f"⚠ No se pudo inicializar AthenaClient ({exc}). Se usará awswrangler.")
else:
    print("⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.")


def athena_query(query: str, database: str = ATHENA_DATABASE) -> pd.DataFrame:
    if ATHENA_MODE == "athena_client" and client is not None:
        return client.query(query)
    return wr.athena.read_sql_query(
        sql=query,
        database=database,
        ctas_approach=False,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_OUTPUT,
    )


def s3_read_csv(path: str, sep: str = "|", **kwargs) -> pd.DataFrame:
    return wr.s3.read_csv(path=path, sep=sep, boto3_session=session, **kwargs)


def test_aws_connection(sample_s3_path: str | None = None) -> None:
    sts = session.client("sts")
    ident = sts.get_caller_identity()
    print(f"✓ AWS Account: {ident.get('Account')} | ARN: {ident.get('Arn')}")

    if sample_s3_path:
        _ = wr.s3.read_csv(path=sample_s3_path, sep='|', boto3_session=session, nrows=1)
        print(f"✓ Lectura S3 OK: {sample_s3_path}")


print(f"Modo Athena activo: {ATHENA_MODE}")
print(f"DB: {ATHENA_DATABASE} | WG: {ATHENA_WORKGROUP}")
print(f"credentials.sh en uso: {credentials_file}")
print("Helper Athena: athena_query(query)")
print("Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')")
print("Diagnóstico opcional: test_aws_connection()")

✓ Credenciales cargadas desde: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.
Modo Athena activo: wrangler
DB: disc_comercial | WG: primary
credentials.sh en uso: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
Helper Athena: athena_query(query)
Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')
Diagnóstico opcional: test_aws_connection()


In [7]:
%%time
query = """

-- Colocar query para armar el dataset de entrenamiento
WITH pd AS (
    SELECT DISTINCT
        -- Identificadores
        a.key_value,
        a.cod_cli,
        date_format(
            date_parse(CAST(a.cod_mes AS varchar), '%Y%m') - interval '1' month,
            '%Y%m'
        ) AS codmes_lag1,
        CAST(a.cod_mes AS INTEGER) AS cod_mes,

        -- Fechas
        TRY_CAST(a.fec_constitucion AS DATE) AS fec_constitucion,

        -- Monetarios
        TRY_CAST(a.mto_pas_soles AS DOUBLE) AS mto_pas_soles,
        TRY_CAST(a.imp_trx_abonosefect_6m AS DOUBLE) AS imp_trx_abonosefect_6m,
        TRY_CAST(a.imp_trx_cargosefe_6m AS DOUBLE) AS imp_trx_cargosefe_6m,
        TRY_CAST(a.avg_trx_cargostot_3m AS DOUBLE) AS avg_trx_cargostot_3m,
    TRY_CAST(a.max_trx_abonos_3m AS DOUBLE) AS max_trx_abonos_3m,

        -- Cantidades
        TRY_CAST(a.cnt_trx_cargostot_3m AS INTEGER) AS cnt_trx_cargostot_3m,

        -- Promedios / ratios
        TRY_CAST(a.cnt_trx_abonospromtot_3m AS DOUBLE) AS cnt_trx_abonospromtot_3m,
        TRY_CAST(a.rat_trx_abonosefectot_1m AS DOUBLE) AS rat_trx_abonosefectot_1m,
        TRY_CAST(a.rat_trx_abonosefectot_3m AS DOUBLE) AS rat_trx_abonosefectot_3m,
        TRY_CAST(a.rat_trx_abonosefectot_9m AS DOUBLE) AS rat_trx_abonosefectot_9m,
        TRY_CAST(a.rat_mntcrgsefetot_1m AS DOUBLE) AS rat_mntcrgsefetot_1m,

        -- Demográficas / antigüedad
        TRY_CAST(a.num_edad_constitucion AS INTEGER) AS num_edad_constitucion,
    TRY_CAST(a.num_antiguedad AS INTEGER) AS num_antiguedad,

        -- Riesgo
        TRY_CAST(a.desc_nivel_rsg_lsb_tot AS DOUBLE) AS desc_nivel_rsg_lsb_tot,

        -- Actividad mensual
        TRY_CAST(a.cnt_meses_siningresos_12m AS INTEGER) AS cnt_meses_siningresos_12m,
        TRY_CAST(a.cnt_meses_sinegresos_12m AS INTEGER) AS cnt_meses_sinegresos_12m,

        -- Ubicación / segmentación
        a.desc_provincia,
        a.desc_departamento,
        a.cod_ubigeo_cd,
        a.cod_sectorista_id,
        a.cod_ciiu_v4,

        -- Flags (string/bool → 0/1)
        CAST(a.flg_casos_hist AS INTEGER) AS flg_casos_hist,
        CAST(a.flg_vrcn_abonos_5m_1m AS INTEGER) AS flg_vrcn_abonos_5m_1m,
        CAST(a.flg_vrcn_efe_cargos_5m_1m AS INTEGER) AS flg_vrcn_efe_cargos_5m_1m,

        -- Conteos
        a.cnt_ro_debajo_umbral,
        -- Perfil económico
        a.mto_fact_declarado_sunat,
        TRY_CAST(a.avg_cp_men_ing_12m AS DOUBLE) AS avg_cp_men_ing_12m,
        TRY_CAST(a.avg_cpmenegr_12m AS DOUBLE) AS avg_cpmenegr_12m,
        TRY_CAST(a.max_mto_cpmening_12m AS DOUBLE) AS max_mto_cpmening_12m,
        TRY_CAST(a.max_mto_cpegrmen_12m AS DOUBLE) AS max_mto_cpegrmen_12m,

        -- Exterior
        a.flg_al_ext_12m,
        a.flg_del_ext_12m,
        TRY_CAST(a.cnt_trx_sinenv_alext_12m AS INTEGER) AS cnt_trx_sinenv_alext_12m,
        TRY_CAST(a.cnt_trx_al_ext_1000_12m AS INTEGER) AS cnt_trx_al_ext_1000_12m,
        TRY_CAST(a.mto_al_ext_12m AS DOUBLE) AS mto_al_ext_12m,
        TRY_CAST(a.mto_del_ext_12m AS DOUBLE) AS mto_del_ext_12m,

        -- Reputacional / antecedentes
        a.flg_pep,
        a.cod_rsg_pep,
        a.flg_activo_pep,
        TRY_CAST(a.cnt_noticias AS INTEGER) AS cnt_noticias,
        a.flg_ros_12m,
        a.flg_alerta_12m,
        TRY_CAST(a.cnt_alerta_hist AS INTEGER) AS cnt_alerta_hist,
        TRY_CAST(a.cnt_ros_hist AS INTEGER) AS cnt_ros_hist,

        -- KYC
        a.flg_kyc_12m,
        a.flg_kyc_hist,
        TRY_CAST(a.cnt_kyc_hist AS INTEGER) AS cnt_kyc_hist,
        -- ======================================================
        -- 🔹 ACELERACIÓN / CAMBIO DE COMPORTAMIENTO
        -- ======================================================
        TRY_CAST(a.imp_trx_abonostot_1m AS DOUBLE)
            / NULLIF(TRY_CAST(a.avg_trx_abonostot_6m AS DOUBLE), 0)
            AS rat_abonos_1m_vs_6m,

        TRY_CAST(a.imp_trx_cargostot_1m AS DOUBLE)
            / NULLIF(TRY_CAST(a.avg_trx_cargostot_6m AS DOUBLE), 0)
            AS rat_cargos_1m_vs_6m,


        -- ======================================================
        -- 🔹 CONCENTRACIÓN EN CONTRAPARTE
        -- ======================================================
        TRY_CAST(a.avg_cpmenegr_12m AS DOUBLE)
            / NULLIF(TRY_CAST(a.imp_trx_cargostot_6m AS DOUBLE), 0)
            AS share_cp_egresos,

        TRY_CAST(a.avg_cp_men_ing_12m AS DOUBLE)
            / NULLIF(TRY_CAST(a.imp_trx_abonostot_6m AS DOUBLE), 0)
            AS share_cp_ingresos,


        -- ======================================================
        -- 🔹 NORMALIZACIÓN DE RIESGO
        -- ======================================================
        TRY_CAST(a.cnt_ros_hist AS DOUBLE)
            / NULLIF(TRY_CAST(a.cnt_trx_cargostot_3m AS DOUBLE), 0)
            AS rat_cntros_x_cnttrxegr_3m,

        TRY_CAST(a.cnt_alerta_hist AS DOUBLE)
            / NULLIF(TRY_CAST(a.num_antiguedad AS DOUBLE), 0)
            AS rat_cnt_alert_x_num_antig,


        -- ======================================================
        -- 🔹 COHERENCIA ECONÓMICA
        -- ======================================================
        TRY_CAST(a.imp_trx_abonostot_6m AS DOUBLE)
            / NULLIF(TRY_CAST(a.mto_fact_declarado_sunat AS DOUBLE), 0)
            AS rat_ing_tot_x_factura_6m,

        TRY_CAST(a.mto_pas_soles AS DOUBLE)
            / NULLIF(TRY_CAST(a.imp_trx_abonostot_6m AS DOUBLE), 0)
            AS rat_pastot_x_ingtot_6m,



        -- ======================================================
        -- 🔹 EXPOSICIÓN AL EXTERIOR (PROPORCIONES)
        -- ======================================================
        TRY_CAST(a.mto_al_ext_12m AS DOUBLE)
            / NULLIF(TRY_CAST(a.imp_trx_abonostot_12m AS DOUBLE), 0)
            AS ratio_egresos_exterior,

        TRY_CAST(a.mto_del_ext_12m AS DOUBLE)
            / NULLIF(TRY_CAST(a.imp_trx_abonostot_12m AS DOUBLE), 0)
            AS rat_ing_ext_x_ing_tot_12m,


        -- ======================================================
        -- 🔹 COHERENCIA PEP / LSB
        -- ======================================================
        TRY_CAST(a.cod_rsg_pep AS DOUBLE)
            - TRY_CAST(a.desc_nivel_rsg_lsb_tot AS DOUBLE)
            AS gap_riesgo_pep_lsb

    FROM d_perm_aws.t_agg_alertas_plaft a
        WHERE a.cod_mes BETWEEN '202501' AND '202604'
            AND a.desc_subsegmento = 'BPE'
),

target AS (
    SELECT 
        codunico,
        periodo_alerta,
        tipo_alerta_n2 AS tip_alerta,
        trx_riesgo_cliente,
        MAX(calificacion_monitoreo) AS flg_alerta
    FROM e_perm_aws.t_alertas_plaft
    GROUP BY codunico, periodo_alerta, tipo_alerta_n2, trx_riesgo_cliente
)

SELECT 
    a.*,
    b.tip_alerta,
    b.trx_riesgo_cliente, 
    CASE 
        WHEN b.flg_alerta = '1' THEN 1 
        ELSE 0 
    END AS target_m
FROM pd a
LEFT JOIN target b
    ON a.cod_cli = b.codunico
    AND cast(a.cod_mes as varchar) = b.periodo_alerta
--   AND codmes_lag1 = c.periodo_alerta
;"""
df_dataset = athena_query(query, database='disc_comercial')
df_dataset.head()

CPU times: total: 40.6 s
Wall time: 10min 41s


,key_value,cod_cli,codmes_lag1,cod_mes,fec_constitucion,mto_pas_soles,imp_trx_abonosefect_6m,imp_trx_cargosefe_6m,avg_trx_cargostot_3m,max_trx_abonos_3m,...,rat_cntros_x_cnttrxegr_3m,rat_cnt_alert_x_num_antig,rat_ing_tot_x_factura_6m,rat_pastot_x_ingtot_6m,ratio_egresos_exterior,rat_ing_ext_x_ing_tot_12m,gap_riesgo_pep_lsb,tip_alerta,trx_riesgo_cliente,target_m
0,5191A59E718ABE48167E86C2ABA6072031BD2F4503BCF8...,0021289654,202507,202508,2018-04-24,0.00,0.00,0.00,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,0
1,4CCD0B1C0E580111C8D3A8A6DAD5F751206166A5F6DA44...,0012320248,202512,202601,2011-05-03,167371.00,40975.00,2000.00,0.00,100218.03,...,NaN,NaN,NaN,1.13,NaN,0.49,NaN,<NA>,<NA>,0
2,68777EFAB7BC981109A7EFCA3999EB1D8D1CB868B60864...,0017266304,202512,202601,2019-09-16,69907.17,0.00,0.00,0.00,10871.57,...,NaN,NaN,NaN,1.34,NaN,NaN,NaN,<NA>,<NA>,0
3,0883096CC972DD192BE667A6B442A46CD83A73BCA11046...,0013346140,202508,202509,2008-07-09,565.06,0.00,0.00,0.00,1000.00,...,NaN,NaN,0.01,0.19,NaN,NaN,NaN,<NA>,<NA>,0
4,3985E0ACE6431ABE1462D4659689712734F9A753C930DF...,0021283172,202508,202509,2021-03-10,264981.39,0.00,0.00,0.00,223000.00,...,NaN,NaN,NaN,1.07,NaN,NaN,NaN,<NA>,<NA>,0


In [8]:
df= df_dataset

In [9]:
df_1=df

In [10]:
df_2 = df_1

In [11]:
# Renombrar y reordenar columnas
df_2 = df_2.rename(columns={'target_m': 'target'})
df_2 =df_2[['target'] + [c for c in df_2.columns if c != 'target']]


In [13]:
df_2["tip_alerta"] =df_2["tip_alerta"].fillna(0)

In [14]:
df_2 = df_2.drop_duplicates(subset=['key_value', 'cod_mes'], keep='first')

In [15]:
import pandas as pd
import numpy as np

# ==========================================
# 1. Copiar DF original
# ==========================================
df_3 = df_2.copy()

# ==========================================
# 3. Columnas Int32 → rellenar NA → convertir a int64
# ==========================================
int32_cols = df_3.select_dtypes(include=["Int32"]).columns

df_3[int32_cols] = df_3[int32_cols].fillna(0).astype("int64")

# ==========================================
# 4. Columnas float → rellenar NA con 0
# ==========================================
float_cols = df_3.select_dtypes(include=["float64", "Float64"]).columns

df_3[float_cols] = df_3[float_cols].fillna(0)

# ==========================================
# 5. Columnas boolean → rellenar NA con False
# ==========================================
bool_cols = df_3.select_dtypes(include=["boolean"]).columns

df_3[bool_cols] = df_3[bool_cols].fillna(False)

# ==========================================
# 6. Columnas categóricas (strings) → NA = "SIN_INFO"
# ==========================================
cat_cols = df_3.select_dtypes(include=["object", "string"]).columns

df_3[cat_cols] = df_3[cat_cols].fillna("SIN_INFO")


In [16]:
df_3[['cod_mes', 'target']].value_counts()

cod_mes  target
202604   0         172268
202512   0         171315
202601   0         170948
202603   0         170298
202602   0         168996
202511   0         166253
202510   0         164651
202507   0         163471
202506   0         162767
202509   0         162739
202508   0         161297
202505   0         158189
202504   0         157311
202503   0         155869
202502   0         155562
202501   0         155135
202604   1            237
202602   1            152
202505   1            147
202502   1            145
202503   1            135
202603   1            128
202501   1            125
202510   1            123
202601   1            122
202508   1            118
202506   1            116
202504   1            108
202509   1             99
202507   1             96
202511   1             64
202512   1             16
Name: count, dtype: int64

In [18]:
import pandas as pd
from sklearn.utils import resample

# ===========================
# 1️⃣ Separar antiguos y recientes
# ===========================
df_antiguos = df_3[df_3['cod_mes'] <= 202507].copy()
df_recientes = df_3[df_3['cod_mes'].between(202508, 202604)].copy()
#df_recientes = df_3[df_3['cod_mes'] >= 202508].copy()  # Test completo

# ===========================
# 2️⃣ Separar clases en antiguos
# ===========================
df_antiguos_con_alerta = df_antiguos[df_antiguos['tip_alerta'] != "0"]  # Con alerta
df_antiguos_sin_alerta = df_antiguos[df_antiguos['tip_alerta'] == "0"]  # Sin alerta

# ===========================
# 3️⃣ Filtrar clases target == 1 (minoritarios)
# ===========================
df_antiguos_1 = df_antiguos_con_alerta[df_antiguos_con_alerta['target'] == 1]  # Alerta y target == 1
df_antiguos_0 = df_antiguos_con_alerta[df_antiguos_con_alerta['target'] == 0]  # Alerta y target == 0

# ===========================
# 4️⃣ Balanceo de clases en TRAIN
# ===========================
n_pos = len(df_antiguos_1)  # Cantidad de clases 1 (minoritarias)
n_neg_deseado = int(n_pos * (99.5 / 0.5))  # Queremos 99.5% de clase 0

# Ajustar clase 0 (target == 0)
if n_neg_deseado <= len(df_antiguos_0):
    df_antiguos_0_bal = resample(df_antiguos_0, replace=False, n_samples=n_neg_deseado, random_state=42)
else:
    df_antiguos_0_bal = resample(df_antiguos_0, replace=True, n_samples=n_neg_deseado, random_state=42)

# ===========================
# 5️⃣ Combinar con clases 1 (target == 1)
# ===========================
df_antiguos_1_bal = df_antiguos_1  # Mantener todas las clases 1 (no se ajustan, ya que se mantiene su porcentaje)

# Combinar 0 (balanceado) y 1 (original) para el dataset de entrenamiento
df_antiguos_balanceados = pd.concat([df_antiguos_0_bal, df_antiguos_1_bal], axis=0)

# ===========================
# 6️⃣ Tomamos el 0.5% de los SIN alerta
# ===========================
# Para tener el 99.5% de clase 0 y 0.5% de clase 1 en el TRAIN
porcentaje_sin_alerta = 0.005  # 0.5% de los registros sin alerta

# Tomamos el 0.5% de los casos sin alerta (sin alertas)
df_sin_alerta_sample = df_antiguos_sin_alerta.sample(frac=porcentaje_sin_alerta, random_state=42)

# ===========================
# 7️⃣ Concatenar alertas + 0.5% de sin alerta
# ===========================
df_train = pd.concat([df_antiguos_balanceados, df_sin_alerta_sample], axis=0)

# Barajamos los datos para evitar sesgo de orden
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

# ===========================
# 8️⃣ Mantener df_test intacto
# ===========================
# Filtrar df_test para asegurarnos de que tiene solo los registros del test (cod_mes >= 202508)
df_test=df_3[df_3['cod_mes'].between(202508, 202604)].copy()
#df_test = df_3[df_3['cod_mes'] >= 202508].copy()

# ===========================
# 9️⃣ Combinar df_train y df_test para df_7
# ===========================
df_4 = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)

# ===========================
# 10️⃣ Borrar columna 'tipo_alerta_n2' de df_7
# ===========================
#df_4 = df_4.drop(columns=['tipo_alerta_n2'], errors='ignore')

# ===========================
# 11️⃣ Verificar distribución final
# ===========================
print("Distribución final de clases en df_train (target):")
print(df_train['target'].value_counts(normalize=True))

print("\nDistribución final de tipo_alerta_n2 en df_train (Eliminada):")
print(df_train['tip_alerta'].value_counts())

print("\nDistribución final de clases en df_test (target):")
print(df_test['target'].value_counts(normalize=True))

print("\nDistribución final de clases en df_7 (target):")
print(df_4['target'].value_counts(normalize=True))

Distribución final de clases en df_train (target):
target
0   0.99
1   0.01
Name: proportion, dtype: float64

Distribución final de tipo_alerta_n2 en df_train (Eliminada):
tip_alerta
0                  173001
AUTOMATICA            816
MANUAL                415
SEMI AUTOMATICA       168
Name: count, dtype: int64

Distribución final de clases en df_test (target):
target
0   1.00
1   0.00
Name: proportion, dtype: float64

Distribución final de clases en df_7 (target):
target
0   1.00
1   0.00
Name: proportion, dtype: float64


In [19]:
df_5 = df_4

In [20]:
df_5[['cod_mes', 'target']].value_counts()

cod_mes  target
202604   0         172268
202512   0         171315
202601   0         170948
202603   0         170298
202602   0         168996
202511   0         166253
202510   0         164651
202509   0         162739
202508   0         161297
202506   0          25632
202507   0          25603
202505   0          24807
202503   0          24610
202504   0          24575
202502   0          24261
202501   0          24040
202604   1            237
202602   1            152
202505   1            147
202502   1            145
202503   1            135
202603   1            128
202501   1            125
202510   1            123
202601   1            122
202508   1            118
202506   1            116
202504   1            108
202509   1             99
202507   1             96
202511   1             64
202512   1             16
Name: count, dtype: int64

In [21]:
# Eliminar columnas 'fec_constitucion' si existe
cols_a_eliminar = ["fec_constitucion"]
df_5 = df_5.drop(columns=cols_a_eliminar, errors="ignore")

In [23]:
df_5.tip_alerta.value_counts()

tip_alerta
0                  1676444
AUTOMATICA            5472
MANUAL                1622
SEMI AUTOMATICA        686
Name: count, dtype: int64

In [24]:
df_5['tip_alerta'] = df_5['tip_alerta'].astype(str)

In [25]:
df_5["cnt_alerta_hist"] = (
    df_5["cnt_alerta_hist"]
    .astype("float64")
)

In [26]:
df_5["mto_fact_declarado_sunat"] = pd.to_numeric(df_5["mto_fact_declarado_sunat"], errors="coerce")

In [27]:
df_5.to_parquet(
    's3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/DATA_INFERENCIA/data_pn_total_expandido_new.parquet',
    index=False
)

In [24]:
df_5.head()

,target,num_documento,codunico,codmes_lag1,mes_base,pasivo_soles,trx_monto_abonos_6m_efectivo,trx_monto_cargos_6m_efectivo,trx_monto_cargos_promedio_3m_total,trx_monto_abonos_3m_max,...,share_cp_egresos,share_cp_ingresos,ros_por_trx_3m,alertas_por_antiguedad,ingresos_vs_facturacion,pasivo_vs_ingresos,ratio_egresos_exterior,ratio_ingresos_exterior,gap_riesgo_pep_lsb,tipo_alerta_n2
0,0,42F5EB1FDB77CAEAE9EC87F623FF94B6A0EA1CA730278A...,0020365850,202502,202503,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
1,0,4D387B86B99EA7D8B7383ED1688A013A4C54D7DD6FBD9D...,0021175178,202412,202501,0.20,0.00,0.00,0.00,5119.00,...,0.00,0.00,0.00,0.00,4762.73,0.00,0.00,0.00,0.00,0
2,0,7D9A0457E9043E163DB108E21E3573D9B5FE2AF1BB4848...,0020545801,202412,202501,0.70,221648.00,0.00,0.00,221648.00,...,0.00,0.00,0.00,0.00,73882.67,0.00,0.00,0.00,0.00,0
3,0,800366DFB9A5FFD355EC0199963AE9100535BAD20532D8...,0021410382,202503,202504,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
4,0,531A21C55C0F088B0B460561EBFA0F0D5D861F0E14256E...,0018660057,202502,202503,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,1119.02,0.00,0.00,0.00,0.00,0


In [28]:
list(df_5)

['target',
 'key_value',
 'cod_cli',
 'codmes_lag1',
 'cod_mes',
 'mto_pas_soles',
 'imp_trx_abonosefect_6m',
 'imp_trx_cargosefe_6m',
 'avg_trx_cargostot_3m',
 'max_trx_abonos_3m',
 'cnt_trx_cargostot_3m',
 'cnt_trx_abonospromtot_3m',
 'rat_trx_abonosefectot_1m',
 'rat_trx_abonosefectot_3m',
 'rat_trx_abonosefectot_9m',
 'rat_mntcrgsefetot_1m',
 'num_edad_constitucion',
 'num_antiguedad',
 'desc_nivel_rsg_lsb_tot',
 'cnt_meses_siningresos_12m',
 'cnt_meses_sinegresos_12m',
 'desc_provincia',
 'desc_departamento',
 'cod_ubigeo_cd',
 'cod_sectorista_id',
 'cod_ciiu_v4',
 'flg_casos_hist',
 'flg_vrcn_abonos_5m_1m',
 'flg_vrcn_efe_cargos_5m_1m',
 'cnt_ro_debajo_umbral',
 'mto_fact_declarado_sunat',
 'avg_cp_men_ing_12m',
 'avg_cpmenegr_12m',
 'max_mto_cpmening_12m',
 'max_mto_cpegrmen_12m',
 'flg_al_ext_12m',
 'flg_del_ext_12m',
 'cnt_trx_sinenv_alext_12m',
 'cnt_trx_al_ext_1000_12m',
 'mto_al_ext_12m',
 'mto_del_ext_12m',
 'flg_pep',
 'cod_rsg_pep',
 'flg_activo_pep',
 'cnt_noticias',
 '

In [ ]:
# Separar train/test por cod_mes
from sklearn.preprocessing import LabelEncoder
df_train = df_5[df_5["cod_mes"] < 202508].copy()
df_test  = df_5[df_5["cod_mes"] >= 202508].copy()

# Concatenar para LabelEncoding
categoricas = [ 
                "desc_provincia","cnt_ro_debajo_umbral",
               "desc_departamento", "cod_ubigeo_cd", "cod_sectorista_id", "cod_ciiu_v4","tipo_alerta_n2"]

df_all = pd.concat([df_train, df_test], axis=0)

for c in categoricas:
    le = LabelEncoder()
    df_all[c] = le.fit_transform(df_all[c].astype(str))

# Separar nuevamente
df_train = df_all.loc[df_all["cod_mes"] < 202508].copy()
df_test  = df_all.loc[df_all["cod_mes"] >= 202508].copy()

# Ahora puedes borrar columnas no deseadas
cols_drop = ["cod_mes","key_value"]
df_train = df_train.drop(columns=cols_drop, errors="ignore")
df_test  = df_test.drop(columns=cols_drop, errors="ignore")

In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, precision_score


# ========================================
# 1️⃣ Separar features y target
# ========================================


target = "target"
cols_excluir = ["tipo_alerta_n2"]

X_train = df_train.drop(columns=[target] + cols_excluir)
#X_train = df_train.drop(columns=[target])
y_train = df_train[target]
#X_test  = df_test.drop(columns=[target] )
X_test  = df_test.drop(columns=[target] + cols_excluir)
y_test  = df_test[target]

# ========================================
# 2️⃣ Columnas categóricas y numéricas
# ========================================
categoricas = X_train.select_dtypes(include=["object", "string", "bool"]).columns.tolist()
numericas   = [c for c in X_train.columns if c not in categoricas]

# ========================================
# 3️⃣ OneHot transformer
# ========================================
onehot = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), categoricas)],
    remainder="passthrough"
)

# ========================================
# 4️⃣ Pipeline XGBoost
# ========================================
pipeline_xgb = Pipeline(steps=[
    ("onehot", onehot),
    ("xgb", XGBClassifier(
        eval_metric="logloss",
        tree_method="hist",
        use_label_encoder=False,
        random_state=42
    ))
])

# ========================================
# 5️⃣ Espacio HPO
# ========================================
from scipy.stats import randint, uniform

param_dist = {
    "xgb__max_depth": randint(3, 8),
    "xgb__learning_rate": uniform(0.01, 0.15),
    "xgb__subsample": uniform(0.6, 0.4),
    "xgb__colsample_bytree": uniform(0.6, 0.4),
    "xgb__gamma": uniform(0, 5),
    "xgb__min_child_weight": randint(1, 10),
    "xgb__n_estimators": randint(200, 800)
}

# ========================================
# 6️⃣ Cross-validation
# ========================================
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# ========================================
# 7️⃣ RandomizedSearchCV para PRECISIÓN
# ========================================
search_xgb = RandomizedSearchCV(
    estimator=pipeline_xgb,
    param_distributions=param_dist,
    n_iter=15,          # menos iteraciones para acelerar
    scoring="precision",
    cv=cv,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

# ========================================
# 8️⃣ Entrenar
# ========================================
search_xgb.fit(X_train, y_train)

# ========================================
# 9️⃣ Resultados HPO
# ========================================
print("🏆 Mejores hiperparámetros para maximizar PRECISIÓN:")
print(search_xgb.best_params_)

# ========================================
# 🔟 Evaluar sobre test
# ========================================
y_pred_prob = search_xgb.predict_proba(X_test)[:,1]

# Métricas
auc = roc_auc_score(y_test, y_pred_prob)
gini = 2*auc - 1

# KS Score
def ks_score(y_true, y_score):
    df = pd.DataFrame({"y": y_true, "score": y_score})
    df = df.sort_values("score", ascending=False)
    df["cum_event"] = (df["y"]==1).cumsum() / df["y"].sum()
    df["cum_nonevent"] = (df["y"]==0).cumsum() / (len(df) - df["y"].sum())
    ks = (df["cum_event"] - df["cum_nonevent"]).max()
    return ks

ks = ks_score(y_test, y_pred_prob)

print(f"\n📊 Métricas en TEST:")
print(f"AUC  : {auc:.4f}")
print(f"Gini : {gini:.4f}")
print(f"KS   : {ks:.4f}")


Fitting 3 folds for each of 15 candidates, totalling 45 fits


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the

🏆 Mejores hiperparámetros para maximizar PRECISIÓN:
{'xgb__colsample_bytree': 0.6232334448672797, 'xgb__gamma': 4.330880728874676, 'xgb__learning_rate': 0.10016725176148131, 'xgb__max_depth': 5, 'xgb__min_child_weight': 6, 'xgb__n_estimators': 508, 'xgb__subsample': 0.9879639408647978}

📊 Métricas en TEST:
AUC  : 0.9626
Gini : 0.9253
KS   : 0.8266


In [27]:
from sklearn.calibration import CalibratedClassifierCV

In [28]:
# Mejor modelo del HPO
best_model = search_xgb.best_estimator_

# Calibración (isotonic)
calibrated_model = CalibratedClassifierCV(
    estimator=best_model,
    method="isotonic",
    cv=3  # CV interno SOLO para calibrar
)

# Entrenar calibración
calibrated_model.fit(X_train, y_train)


,estimator,"Pipeline(step...=None, ...))])"
,method,'isotonic'
,cv,3
,n_jobs,None
,ensemble,'auto'
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False


[CV] END xgb__colsample_bytree=0.749816047538945, xgb__gamma=4.75357153204958, xgb__learning_rate=0.11979909127171076, xgb__max_depth=7, xgb__min_child_weight=5, xgb__n_estimators=321, xgb__subsample=0.662397808134481; total time=  20.0s
[CV] END xgb__colsample_bytree=0.9329770563201687, xgb__gamma=1.0616955533913808, xgb__learning_rate=0.03727374508106509, xgb__max_depth=7, xgb__min_child_weight=1, xgb__n_estimators=659, xgb__subsample=0.8446612641953124; total time= 2.0min
[CV] END xgb__colsample_bytree=0.6028265220878869, xgb__gamma=0.11531212520707879, xgb__learning_rate=0.08871619903875837, xgb__max_depth=4, xgb__min_child_weight=3, xgb__n_estimators=766, xgb__subsample=0.9932923543227152; total time=  59.2s
[CV] END xgb__colsample_bytree=0.8733054075301833, xgb__gamma=3.0499832889131047, xgb__learning_rate=0.13497923676042464, xgb__max_depth=5, xgb__min_child_weight=1, xgb__n_estimators=761, xgb__subsample=0.8650089137415928; total time=  35.9s
[CV] END xgb__colsample_bytree=0.72

In [36]:
y_pred_prob_cal = calibrated_model.predict_proba(X_test)[:, 1]

In [37]:
auc_cal = roc_auc_score(y_test, y_pred_prob_cal)
gini_cal = 2 * auc_cal - 1
ks_cal = ks_score(y_test, y_pred_prob_cal)

print("\n📊 Métricas en TEST (CALIBRADO):")
print(f"AUC  : {auc_cal:.4f}")
print(f"Gini : {gini_cal:.4f}")
print(f"KS   : {ks_cal:.4f}")


📊 Métricas en TEST (CALIBRADO):
AUC  : 0.9581
Gini : 0.9162
KS   : 0.8225


In [38]:
# ==============================================================
# 7) Generar deciles y micro-segmentación (CORREGIDO)
# ==============================================================

# Asegurar que df_test tenga los scores del modelo
df_eval = df_test.copy()
df_eval['score'] = y_pred_prob_cal  # score del mejor modelo


# ==============================================================
# 7a) DECILES (10 grupos)
# ==============================================================
df_eval["decile"] = pd.qcut(df_eval["score"].rank(method="first"), 10, labels=False) + 1

tabla_deciles = df_eval.groupby("decile").agg(
    count=("target", "size"),
    events=("target", "sum"),
    score_mean=("score", "mean")
).reset_index()

tabla_deciles["event_rate"] = tabla_deciles["events"] / tabla_deciles["count"]
tabla_deciles["precision"] = tabla_deciles["event_rate"]

# ORDEN CORRECTO: mayor score → menor score
tabla_deciles = tabla_deciles.sort_values("score_mean", ascending=False).reset_index(drop=True)

# RECALL ACUMULADO
tabla_deciles["recall"] = tabla_deciles["events"].cumsum() / tabla_deciles["events"].sum()

tabla_deciles["lift"] = tabla_deciles["event_rate"] / df_eval["target"].mean()

print("\n📊 TABLA POR DECILES:")
print(tabla_deciles)


📊 TABLA POR DECILES:
   decile  count  events  score_mean  event_rate  precision  recall  lift
0      10  80972     372        0.03        0.00       0.00    0.92  9.21
1       9  80971      15        0.00        0.00       0.00    0.96  0.37
2       8  80971       3        0.00        0.00       0.00    0.97  0.07
3       7  80972       2        0.00        0.00       0.00    0.97  0.05
4       6  80971       2        0.00        0.00       0.00    0.98  0.05
5       5  80971       4        0.00        0.00       0.00    0.99  0.10
6       4  80972       3        0.00        0.00       0.00    0.99  0.07
7       3  80971       0        0.00        0.00       0.00    0.99  0.00
8       2  80971       2        0.00        0.00       0.00    1.00  0.05
9       1  80972       1        0.00        0.00       0.00    1.00  0.02


In [39]:
# ==============================================================
# 7) Generar deciles y micro-segmentación
# ==============================================================

# Asegurar que df_test tenga los scores del modelo
df_eval = df_test.copy()
df_eval["score"] = y_pred_prob_cal  # score del mejor modelo

# ==============================================================
# 🔹 FILTRO SOLO PARA EVALUACIÓN
# ==============================================================
df_eval = df_eval[df_eval["tipo_alerta_n2"] != 0].copy()

# ==============================================================
# 7a) DECILES (10 grupos)
# ==============================================================
df_eval["decile"] = (
    pd.qcut(
        df_eval["score"].rank(method="first"),
        5,
        labels=False
    ) + 1
)

tabla_deciles = df_eval.groupby("decile").agg(
    count=("target", "size"),
    events=("target", "sum"),
    score_mean=("score", "mean")
).reset_index()

tabla_deciles["event_rate"] = tabla_deciles["events"] / tabla_deciles["count"]
tabla_deciles["precision"] = tabla_deciles["event_rate"]

# ORDEN CORRECTO: mayor score → menor score
tabla_deciles = (
    tabla_deciles
    .sort_values("score_mean", ascending=False)
    .reset_index(drop=True)
)

# RECALL ACUMULADO
tabla_deciles["recall"] = (
    tabla_deciles["events"].cumsum()
    / tabla_deciles["events"].sum()
)

tabla_deciles["lift"] = (
    tabla_deciles["event_rate"]
    / df_eval["target"].mean()
)

print("\n📊 TABLA POR DECILES (TEST FILTRADO):")
print(tabla_deciles)



📊 TABLA POR DECILES (TEST FILTRADO):
   decile  count  events  score_mean  event_rate  precision  recall  lift
0       5    708     163        0.29        0.23       0.23    0.40  2.02
1       4    707      95        0.09        0.13       0.13    0.64  1.18
2       3    707      46        0.03        0.07       0.07    0.75  0.57
3       2    707      40        0.01        0.06       0.06    0.85  0.50
4       1    708      60        0.00        0.08       0.08    1.00  0.74


In [40]:
df_5.tipo_alerta_n2.value_counts()

tipo_alerta_n2
0                  976149
AUTOMATICA           3266
MANUAL               1272
SEMI AUTOMATICA       427
Name: count, dtype: int64

In [41]:
import pandas as pd
import numpy as np

def tabla_quintiles_por_mes(df_mes, modelo, target_col="target"):
    if df_mes.empty:
        return None

    # Validación mínima
    if df_mes[target_col].nunique() < 2:
        return None

    # Features
    X = df_mes.drop(columns=[target_col])
    y = df_mes[target_col]

    # Score del modelo
    score = modelo.predict_proba(X)[:, 1]

    df_temp = df_mes.copy()
    df_temp["score"] = score

    # ==========================================================
    # QUINTILES (1 = mayor riesgo)
    # ==========================================================
    df_temp["quintil"] = (
        pd.qcut(
            df_temp["score"].rank(method="first"),
            5,
            labels=False,
            duplicates="drop"
        ) + 1
    )

    # ==========================================================
    # TABLA RESUMEN
    # ==========================================================
    tabla = (
        df_temp
        .groupby("quintil")
        .agg(
            count=(target_col, "size"),
            events=(target_col, "sum"),
            score_mean=("score", "mean")
        )
        .reset_index()
        .sort_values("score_mean", ascending=False)
        .reset_index(drop=True)
    )

    tabla["event_rate"] = tabla["events"] / tabla["count"]

    # Recall acumulado (desde mayor score)
    tabla["recall"] = tabla["events"].cumsum() / tabla["events"].sum()

    # Lift
    tasa_global = df_temp[target_col].mean()
    tabla["lift"] = tabla["event_rate"] / tasa_global

    return tabla


In [42]:
# ==========================================================
# FILTRO BASE (igual que antes)
# ==========================================================
df_all = df_all[df_all["tipo_alerta_n2"] != 0].copy()

meses = [202508, 202509, 202510, 202511, 202512]

salidas = {}

for m in meses:
    df_mes = df_all[df_all["mes_base"] == m].copy()

    tabla = tabla_quintiles_por_mes(
        df_mes,
        search_xgb,   # tu modelo entrenado
        target_col="target"
    )

    salidas[m] = tabla


In [44]:
for m, tabla in salidas.items():
    print(f"\n📌 Resultados para el mes {m}")
    if tabla is not None:
        display(tabla)
    else:
        print("⚠️ No se pudo calcular (sin eventos o datos insuficientes)")



📌 Resultados para el mes 202508


,quintil,count,events,score_mean,event_rate,recall,lift
0,5,108,53,0.45,0.49,0.47,2.35
1,4,108,28,0.12,0.26,0.72,1.24
2,3,108,12,0.04,0.11,0.82,0.53
3,2,108,11,0.01,0.10,0.92,0.49
4,1,108,9,0.00,0.08,1.00,0.40



📌 Resultados para el mes 202509


,quintil,count,events,score_mean,event_rate,recall,lift
0,5,158,35,0.31,0.22,0.36,1.82
1,4,158,19,0.08,0.12,0.56,0.99
2,3,157,16,0.03,0.10,0.73,0.84
3,2,158,14,0.01,0.09,0.88,0.73
4,1,158,12,0.00,0.08,1.00,0.62



📌 Resultados para el mes 202510


,quintil,count,events,score_mean,event_rate,recall,lift
0,5,254,47,0.32,0.19,0.40,1.99
1,4,254,21,0.07,0.08,0.58,0.89
2,3,253,20,0.02,0.08,0.75,0.85
3,2,254,11,0.01,0.04,0.84,0.47
4,1,254,19,0.00,0.07,1.00,0.80



📌 Resultados para el mes 202511


,quintil,count,events,score_mean,event_rate,recall,lift
0,5,87,17,0.33,0.20,0.27,1.34
1,4,86,14,0.08,0.16,0.49,1.12
2,3,87,10,0.03,0.11,0.65,0.79
3,2,86,12,0.01,0.14,0.84,0.96
4,1,87,10,0.00,0.11,1.00,0.79



📌 Resultados para el mes 202512


,quintil,count,events,score_mean,event_rate,recall,lift
0,5,101,9,0.23,0.09,0.64,3.22
1,4,101,3,0.04,0.03,0.86,1.07
2,3,101,0,0.01,0.00,0.86,0.00
3,2,101,2,0.00,0.02,1.00,0.72
4,1,102,0,0.00,0.00,1.00,0.00


In [166]:
import numpy as np

# ==========================================================
# MESES BASE DE REFERENCIA
# ==========================================================
MESES_REF = [202508, 202509,202510,202511,202512]

df_ref = df_all[
    (df_all["mes_base"].isin(MESES_REF)) &
    (df_all["tipo_alerta_n2"] != 0)
].copy()

# Validación mínima
assert df_ref.shape[0] > 0, "❌ df_ref vacío"
assert df_ref["target"].nunique() > 1, "❌ df_ref tiene una sola clase"

# ==========================================================
# FEATURES / TARGET
# ==========================================================
X_ref = df_ref.drop(columns=["target"])
y_ref = df_ref["target"]

# ==========================================================
# SCORE DEL MODELO
# ==========================================================
df_ref["score"] = search_xgb.predict_proba(X_ref)[:, 1]

# ==========================================================
# CORTES DE QUINTILES (FIJOS)
# ==========================================================
cuts = np.quantile(df_ref["score"], [0.2, 0.4, 0.6, 0.8])

cuts


array([0.04487522, 0.21399416, 0.54792063, 0.90338002])

In [ ]:
# ==========================================================
# MES BASE DE REFERENCIA
# ==========================================================
MES_REF = 202508

df_ref = df_all[
    (df_all["cod_mes"] == MES_REF) &
    (df_all["tipo_alerta_n2"] != 0)
].copy()

# Features
X_ref = df_ref.drop(columns=["target"])
y_ref = df_ref["target"]

# Score con el modelo
df_ref["score"] = search_xgb.predict_proba(X_ref)[:, 1]

# Calcular cortes de quintiles (sobre SCORE)
cuts = np.quantile(df_ref["score"], [0.2, 0.4, 0.6, 0.8])

cuts

array([0.00907485, 0.02574072, 0.09001518, 0.25262128])

In [182]:
def tabla_quintiles_con_cortes(df_mes, modelo, cortes, target_col="target"):
    if df_mes.empty:
        return None

    if df_mes[target_col].nunique() < 2:
        return None

    X = df_mes.drop(columns=[target_col])
    y = df_mes[target_col]

    df_temp = df_mes.copy()
    df_temp["score"] = modelo.predict_proba(X)[:, 1]

    # ==========================================================
    # QUINTILES CON CORTES FIJOS (202508)
    # ==========================================================
    bins = [-np.inf] + list(cortes) + [np.inf]

    df_temp["quintil"] = pd.cut(
        df_temp["score"],
        bins=bins,
        labels=[1, 2, 3, 4, 5],
        include_lowest=True
    ).astype(int)

    # ==========================================================
    # TABLA RESUMEN
    # ==========================================================
    tabla = (
        df_temp
        .groupby("quintil")
        .agg(
            count=(target_col, "size"),
            events=(target_col, "sum"),
            score_mean=("score", "mean")
        )
        .reset_index()
        .sort_values("score_mean", ascending=False)
        .reset_index(drop=True)
    )

    tabla["event_rate"] = tabla["events"] / tabla["count"]

    tabla["recall"] = tabla["events"].cumsum() / tabla["events"].sum()

    tasa_global = df_temp[target_col].mean()
    tabla["lift"] = tabla["event_rate"] / tasa_global

    return tabla


In [ ]:
meses = [ 202510, 202511, 202512]

salidas = {}

for m in meses:
    df_mes = df_all[
        (df_all["cod_mes"] == m) &
        (df_all["tipo_alerta_n2"] != 0)
    ].copy()

    tabla = tabla_quintiles_con_cortes(
        df_mes,
        search_xgb,
        cortes=cuts,
        target_col="target"
    )

    salidas[m] = tabla

In [184]:
for m, tabla in salidas.items():
    print(f"\n📌 Resultados para el mes {m}")
    if tabla is not None:
        display(tabla)
    else:
        print("⚠️ No se pudo calcular")



📌 Resultados para el mes 202510


,quintil,count,events,score_mean,event_rate,recall,lift
0,5,178,48,0.41,0.27,0.40,2.88
1,4,231,16,0.16,0.07,0.54,0.74
2,3,272,18,0.05,0.07,0.69,0.71
3,2,262,11,0.02,0.04,0.78,0.45
4,1,326,26,0.00,0.08,1.00,0.85



📌 Resultados para el mes 202511


,quintil,count,events,score_mean,event_rate,recall,lift
0,5,75,20,0.40,0.27,0.31,1.80
1,4,98,15,0.17,0.15,0.55,1.04
2,3,86,9,0.05,0.10,0.69,0.71
3,2,79,8,0.02,0.10,0.81,0.69
4,1,95,12,0.00,0.13,1.00,0.85



📌 Resultados para el mes 202512


,quintil,count,events,score_mean,event_rate,recall,lift
0,5,51,4,0.38,0.08,0.29,2.83
1,4,85,7,0.15,0.08,0.79,2.98
2,3,98,3,0.05,0.03,1.00,1.11
3,2,146,0,0.02,0.00,1.00,0.00
4,1,126,0,0.00,0.00,1.00,0.00


In [63]:
# Asegurar que df_test tenga los scores del modelo
df_eval = df_test.copy()
df_eval["score"] = y_pred_prob_cal  # score del mejor modelo

# ==============================================================
# 🔹 FILTRO SOLO PARA EVALUACIÓN
# ==============================================================
df_eval = df_eval[df_eval["tipo_alerta_n2"] != 0].copy()

In [ ]:
# ==========================================================
# FILTRO BASE (igual que antes)
# ==========================================================
df_all = df_all[df_all["tipo_alerta_n2"] != 0].copy()

meses = [202508, 202509, 202510, 202511, 202512]

salidas = {}

for m in meses:
    df_mes = df_all[df_all["cod_mes"] == m].copy()

    tabla = tabla_quintiles_por_mes(
        df_mes,
        search_xgb,   # tu modelo entrenado
        target_col="target"
    )

    salidas[m] = tabla

In [59]:
for m, tabla in salidas.items():
    print(f"\n📌 Resultados para el mes {m}")
    if tabla is not None:
        display(tabla)
    else:
        print("⚠️ No se pudo calcular (sin eventos o datos insuficientes)")



📌 Resultados para el mes 202508


,quintil,count,events,score_mean,event_rate,recall,lift
0,5,108,67,0.95,0.62,0.60,3.02
1,4,108,30,0.68,0.28,0.87,1.35
2,3,108,7,0.28,0.06,0.94,0.32
3,2,108,5,0.07,0.05,0.98,0.23
4,1,108,2,0.01,0.02,1.00,0.09



📌 Resultados para el mes 202509


,quintil,count,events,score_mean,event_rate,recall,lift
0,5,158,59,0.90,0.37,0.63,3.13
1,4,158,21,0.60,0.13,0.85,1.12
2,3,157,9,0.31,0.06,0.95,0.48
3,2,158,5,0.12,0.03,1.00,0.27
4,1,158,0,0.03,0.00,1.00,0.00



📌 Resultados para el mes 202510


,quintil,count,events,score_mean,event_rate,recall,lift
0,5,254,66,0.97,0.26,0.56,2.82
1,4,254,23,0.88,0.09,0.76,0.98
2,3,253,13,0.70,0.05,0.87,0.56
3,2,254,7,0.38,0.03,0.93,0.30
4,1,254,8,0.07,0.03,1.00,0.34



📌 Resultados para el mes 202511


,quintil,count,events,score_mean,event_rate,recall,lift
0,5,87,35,0.89,0.40,0.54,2.68
1,4,86,22,0.59,0.26,0.88,1.70
2,3,87,4,0.31,0.05,0.94,0.31
3,2,86,3,0.12,0.03,0.98,0.23
4,1,87,1,0.02,0.01,1.00,0.08



📌 Resultados para el mes 202512
⚠️ No se pudo calcular (sin eventos o datos insuficientes)


In [64]:
df_all = df_all[df_all["tipo_alerta_n2"] != 0].copy()

In [ ]:
meses = [202508, 202509, 202510, 202511,202512]

salidas = {}

for m in meses:
    df_mes = df_all[df_all["cod_mes"] == m].copy()
    tabla = tabla_quintiles_por_mes(df_mes, search_xgb, target_col="target")
    salidas[m] = tabla

In [66]:
for m in salidas:
    print(f"\n📌 Resultados para el mes {m}")
    display(salidas[m])


📌 Resultados para el mes 202508


,quintil,count,events,score_mean,event_rate,recall,lift
0,5,108,67,0.95,0.62,0.60,3.02
1,4,108,30,0.68,0.28,0.87,1.35
2,3,108,7,0.28,0.06,0.94,0.32
3,2,108,5,0.07,0.05,0.98,0.23
4,1,108,2,0.01,0.02,1.00,0.09



📌 Resultados para el mes 202509


,quintil,count,events,score_mean,event_rate,recall,lift
0,5,158,59,0.90,0.37,0.63,3.13
1,4,158,21,0.60,0.13,0.85,1.12
2,3,157,9,0.31,0.06,0.95,0.48
3,2,158,5,0.12,0.03,1.00,0.27
4,1,158,0,0.03,0.00,1.00,0.00



📌 Resultados para el mes 202510


,quintil,count,events,score_mean,event_rate,recall,lift
0,5,254,66,0.97,0.26,0.56,2.82
1,4,254,23,0.88,0.09,0.76,0.98
2,3,253,13,0.70,0.05,0.87,0.56
3,2,254,7,0.38,0.03,0.93,0.30
4,1,254,8,0.07,0.03,1.00,0.34



📌 Resultados para el mes 202511


,quintil,count,events,score_mean,event_rate,recall,lift
0,5,87,35,0.89,0.40,0.54,2.68
1,4,86,22,0.59,0.26,0.88,1.70
2,3,87,4,0.31,0.05,0.94,0.31
3,2,86,3,0.12,0.03,0.98,0.23
4,1,87,1,0.02,0.01,1.00,0.08



📌 Resultados para el mes 202512


None

In [42]:
for m in salidas:
    print(f"\n📌 Resultados para el mes {m}")
    display(salidas[m])



📌 Resultados para el mes 202508


,quintil,count,events,score_mean,event_rate,precision,recall,lift
0,5,108,57,0.48,0.53,0.53,0.51,2.57
1,4,108,23,0.14,0.21,0.21,0.72,1.04
2,3,108,9,0.04,0.08,0.08,0.80,0.41
3,2,108,13,0.01,0.12,0.12,0.92,0.59
4,1,108,9,0.00,0.08,0.08,1.00,0.41



📌 Resultados para el mes 202509


,quintil,count,events,score_mean,event_rate,precision,recall,lift
0,5,158,44,0.35,0.28,0.28,0.47,2.34
1,4,158,11,0.10,0.07,0.07,0.59,0.58
2,3,157,11,0.04,0.07,0.07,0.70,0.59
3,2,158,11,0.01,0.07,0.07,0.82,0.58
4,1,158,17,0.01,0.11,0.11,1.00,0.90



📌 Resultados para el mes 202510


...


📌 Resultados para el mes 202511


,quintil,count,events,score_mean,event_rate,precision,recall,lift
0,5,87,21,0.39,0.24,0.24,0.32,1.61
1,4,86,12,0.12,0.14,0.14,0.51,0.93
2,3,87,10,0.04,0.11,0.11,0.66,0.77
3,2,86,10,0.01,0.12,0.12,0.82,0.77
4,1,87,12,0.00,0.14,0.14,1.00,0.92



📌 Resultados para el mes 202512


None

[CV] END xgb__colsample_bytree=0.6232334448672797, xgb__gamma=4.330880728874676, xgb__learning_rate=0.10016725176148131, xgb__max_depth=5, xgb__min_child_weight=6, xgb__n_estimators=508, xgb__subsample=0.9879639408647978; total time=  17.8s
[CV] END xgb__colsample_bytree=0.9329770563201687, xgb__gamma=1.0616955533913808, xgb__learning_rate=0.03727374508106509, xgb__max_depth=7, xgb__min_child_weight=1, xgb__n_estimators=659, xgb__subsample=0.8446612641953124; total time= 1.9min
[CV] END xgb__colsample_bytree=0.786705157299192, xgb__gamma=4.299702033681603, xgb__learning_rate=0.11204613078816694, xgb__max_depth=3, xgb__min_child_weight=7, xgb__n_estimators=473, xgb__subsample=0.9795542149013333; total time=  15.8s
[CV] END xgb__colsample_bytree=0.786705157299192, xgb__gamma=4.299702033681603, xgb__learning_rate=0.11204613078816694, xgb__max_depth=3, xgb__min_child_weight=7, xgb__n_estimators=473, xgb__subsample=0.9795542149013333; total time=  15.8s
[CV] END xgb__colsample_bytree=0.9862

In [145]:
import numpy as np

def obtener_bins_quintiles_fijos():
    bins = [
        -np.inf,   # Quintil 5 (menor riesgo)
        0.0044,
        0.0117,
        0.05,
        0.2385,
        np.inf     # Quintil 1 (mayor riesgo)
    ]
    return bins




In [50]:
import numpy as np

def obtener_bins_quintiles_fijos():
    bins = [
        -np.inf,   # Quintil 5 (menor riesgo)
        
        0.00784,
        0.0515,
        0.387,
        np.inf     # Quintil 1 (mayor riesgo)
    ]
    return bins

In [51]:
import pandas as pd

def tabla_quintiles_por_mes_bins(df_mes, modelo_calibrado, bins, target_col="target"):
    if df_mes.empty:
        return None

    # Features
    X = df_mes.drop(columns=[target_col])

    # 👉 SCORE CALIBRADO
    score = modelo_calibrado.predict_proba(X)[:, 1]

    df_temp = df_mes.copy()
    df_temp["score"] = score

    # Quintiles fijos (1 = mayor riesgo)
    df_temp["quintil"] = pd.cut(
        df_temp["score"],
        bins=bins,
        labels=[ 4, 3, 2, 1],  # invertido
        include_lowest=True
    ).astype(int)

    tabla = (
        df_temp
        .groupby("quintil")
        .agg(
            cantidad_casos=(target_col, "size"),
            casos_positivos=(target_col, "sum"),
            score_mean=("score", "mean"),
            score_min=("score", "min"),
            score_max=("score", "max")
        )
        .reset_index()
        .sort_values("quintil")
    )

    tabla["precision"] = tabla["casos_positivos"] / tabla["cantidad_casos"]
    tabla["recall"] = tabla["casos_positivos"].cumsum() / tabla["casos_positivos"].sum()
    tabla["lift"] = tabla["precision"] / df_temp[target_col].mean()

    return tabla




In [52]:
bins_quintiles = obtener_bins_quintiles_fijos()

meses = [202508, 202509, 202510, 202511, 202512]
salidas = {}

for m in meses:
    df_mes = df_all[df_all["mes_base"] == m].copy()
    tabla = tabla_quintiles_por_mes_bins(
        df_mes,
        search_xgb,
        bins=bins_quintiles,
        target_col="target"
    )
    salidas[m] = tabla



In [53]:
for m in salidas:
    print(f"\n📌 Resultados para el mes {m}")
    display(salidas[m])


📌 Resultados para el mes 202508


,quintil,cantidad_casos,casos_positivos,score_mean,score_min,score_max,precision,recall,lift
0,1,60,38,0.69,0.39,0.95,0.63,0.43,5.53
1,2,136,23,0.16,0.05,0.37,0.17,0.69,1.48
2,3,242,21,0.02,0.01,0.05,0.09,0.92,0.76
3,4,339,7,0.00,0.00,0.01,0.02,1.00,0.18



📌 Resultados para el mes 202509


,quintil,cantidad_casos,casos_positivos,score_mean,score_min,score_max,precision,recall,lift
0,1,264,59,0.61,0.39,0.97,0.22,0.53,2.49
1,2,454,31,0.18,0.05,0.39,0.07,0.80,0.76
2,3,267,14,0.02,0.01,0.05,0.05,0.93,0.58
3,4,264,8,0.00,0.00,0.01,0.03,1.00,0.34



📌 Resultados para el mes 202510


...


📌 Resultados para el mes 202511


,quintil,cantidad_casos,casos_positivos,score_mean,score_min,score_max,precision,recall,lift
0,1,14,4,0.69,0.39,0.96,0.29,0.31,10.95
1,2,63,6,0.13,0.05,0.37,0.10,0.77,3.65
2,3,146,1,0.02,0.01,0.05,0.01,0.85,0.26
3,4,275,2,0.00,0.00,0.01,0.01,1.00,0.28



📌 Resultados para el mes 202512


None

[CV] END xgb__colsample_bytree=0.749816047538945, xgb__gamma=4.75357153204958, xgb__learning_rate=0.11979909127171076, xgb__max_depth=7, xgb__min_child_weight=5, xgb__n_estimators=321, xgb__subsample=0.662397808134481; total time=   0.0s
[CV] END xgb__colsample_bytree=0.6232334448672797, xgb__gamma=4.330880728874676, xgb__learning_rate=0.10016725176148131, xgb__max_depth=5, xgb__min_child_weight=6, xgb__n_estimators=508, xgb__subsample=0.9879639408647978; total time=   0.0s
[CV] END xgb__colsample_bytree=0.6028265220878869, xgb__gamma=0.11531212520707879, xgb__learning_rate=0.08871619903875837, xgb__max_depth=4, xgb__min_child_weight=3, xgb__n_estimators=766, xgb__subsample=0.9932923543227152; total time=   0.0s
[CV] END xgb__colsample_bytree=0.786705157299192, xgb__gamma=4.299702033681603, xgb__learning_rate=0.11204613078816694, xgb__max_depth=3, xgb__min_child_weight=7, xgb__n_estimators=473, xgb__subsample=0.9795542149013333; total time=   0.0s
[CV] END xgb__colsample_bytree=0.98625

In [154]:
for m in salidas:
    print(f"\n📌 Resultados para el mes {m}")
    display(salidas[m])



📌 Resultados para el mes 202508


,quintil,cantidad_casos,casos_positivos,score_mean,score_min,score_max,precision,recall,lift
0,1,65,39,0.63,0.40,0.94,0.60,0.44,5.24
1,2,139,22,0.16,0.05,0.39,0.16,0.69,1.38
2,3,227,23,0.02,0.01,0.05,0.10,0.94,0.88
3,4,346,5,0.00,0.00,0.01,0.01,1.00,0.13



📌 Resultados para el mes 202509


,quintil,cantidad_casos,casos_positivos,score_mean,score_min,score_max,precision,recall,lift
0,1,236,57,0.60,0.39,0.95,0.24,0.50,2.67
1,2,493,37,0.19,0.05,0.39,0.08,0.83,0.83
2,3,272,13,0.02,0.01,0.05,0.05,0.95,0.53
3,4,248,6,0.00,0.00,0.01,0.02,1.00,0.27



📌 Resultados para el mes 202510


,quintil,cantidad_casos,casos_positivos,score_mean,score_min,score_max,precision,recall,lift
0,1,21,10,0.54,0.40,0.74,0.48,0.17,3.36
1,2,89,31,0.15,0.05,0.36,0.35,0.68,2.46
2,3,132,17,0.02,0.01,0.05,0.13,0.97,0.91
3,4,181,2,0.00,0.00,0.01,0.01,1.00,0.08



📌 Resultados para el mes 202511


,quintil,cantidad_casos,casos_positivos,score_mean,score_min,score_max,precision,recall,lift
0,1,17,5,0.68,0.39,0.95,0.29,0.38,11.27
1,2,60,5,0.14,0.05,0.37,0.08,0.77,3.19
2,3,143,0,0.02,0.01,0.05,0.00,0.77,0.00
3,4,278,3,0.00,0.00,0.01,0.01,1.00,0.41



📌 Resultados para el mes 202512


None

In [85]:
def agregar_score_calibrado(df_mes, modelo_calibrado, target_col="target"):
    if df_mes.empty:
        return None

    X = df_mes.drop(columns=[target_col])
    score = modelo_calibrado.predict_proba(X)[:, 1]

    df_out = df_mes.copy()
    df_out["probabilidad"] = score

    return df_out


In [86]:
from sklearn.metrics import precision_recall_curve

def encontrar_cutoff_optimo(y_true, y_score):
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)

    # Evitar división por cero
    f1 = 2 * (precision * recall) / (precision + recall + 1e-9)

    idx = f1.argmax()

    return {
        "cutoff": thresholds[idx],
        "precision": precision[idx],
        "recall": recall[idx],
        "f1": f1[idx]
    }


In [89]:
meses_futuros = [202509, 202510, 202511]

resultados_cutoff = []

for m in meses_futuros:
    df_mes = df_all[df_all["mes_base"] == m].copy()

    df_eval = agregar_score_calibrado(
        df_mes,
        calibrated_model,
        target_col="target"
    )

    res = encontrar_cutoff_optimo(
        y_true=df_eval["target"],
        y_score=df_eval["probabilidad"]
    )

    res["mes_base"] = m
    resultados_cutoff.append(res)

df_cutoffs = pd.DataFrame(resultados_cutoff)

display(df_cutoffs)


,cutoff,precision,recall,f1,mes_base
0,0.17,0.48,0.63,0.55,202509
1,0.67,0.35,0.55,0.42,202510
2,0.17,0.47,0.50,0.48,202511


In [90]:
def aplicar_cutoff(df, cutoff):
    df = df.copy()
    df["pred"] = (df["probabilidad"] >= cutoff).astype(int)
    return df


In [91]:
df_test_eval = agregar_score_calibrado(df_test, calibrated_model)

cutoff_global = df_cutoffs["cutoff"].median()

df_test_eval = aplicar_cutoff(df_test_eval, cutoff_global)

df_test_eval.groupby("pred").agg(
    casos=("target", "size"),
    eventos=("target", "sum"),
    event_rate=("target", "mean")
)


,casos,eventos,event_rate
pred,,,
0,2199,128,0.06
1,832,257,0.31


In [96]:
list(df_4)

['target',
 'num_documento',
 'mes_base',
 'pasivo_soles',
 'trx_monto_abonos_6m_efectivo',
 'trx_monto_cargos_6m_efectivo',
 'trx_q_cargos_3m_total',
 'trx_q_abonos_promedio_3m_total',
 'trx_monto_cargos_promedio_3m_total',
 'trx_q_abonos_ratio_1m_efectivo_total',
 'trx_q_abonos_ratio_3m_efectivo_total',
 'trx_q_abonos_ratio_9m_efectivo_total',
 'trx_monto_cargos_ratio_1m_efectivo_total',
 'trx_monto_abonos_3m_max',
 'edad_constitucion',
 'antiguedad',
 'nivel_riesgo_lsb_total',
 'q_meses_ingresos_0',
 'q_meses_egresos_0',
 'provincia',
 'departamento',
 'ubigeo_cd',
 'sectorista_id',
 'ciiu_v4',
 'flag_casos_hist',
 'flag_variacion_abono_monto_total_5m_1m',
 'flag_variacion_efect_cargos_monto_5m_1m',
 'q_ro_debajo_umbral',
 'q_alerta_hist',
 'q_ros_hist',
 'tipo_alerta_n2']

In [ ]:
import pandas as pd
import numpy as np

# ==============================================================
# 1) Asegurarnos de tener el score en df_test + variables originales
# ==============================================================
# Suponiendo que df_4 es tu dataframe original completo (con columnas originales)
# y que ya tienes: y_pred_prob = search_xgb.predict_proba(X_test)[:,1]
df_final = df_4[df_4["cod_mes"] == 202508].copy()  # solo período de evaluación
df_final["score"] = y_pred_prob

# Seleccionar solo las variables relevantes del modelo
variables_modelo = [
    "trx_monto_abonos_3m_max", "cod_sectorista_id", "pasivo_soles", "q_meses_ingresos_0",
    "cod_ciiu_v4", "trx_monto_abonos_6m_efectivo", "cnt_meses_sinegresos_12m", "num_antiguedad",
    "trx_monto_cargos_6m_efectivo",
    "trx_q_abonos_promedio_3m_total", "trx_q_abonos_ratio_9m_efectivo_total",
    "edad_constitucion", "cod_ubigeo_cd", "trx_q_abonos_ratio_1m_efectivo_total",
    "trx_monto_cargos_ratio_1m_efectivo_total", "desc_departamento",
    "trx_monto_cargos_promedio_3m_total","tipo_alerta_n2"
]

# Asegurarse de que todas las columnas existan (ignorar si alguna falta)
df_final = df_final[variables_modelo + ["score", "target", "key_value"]].copy()

# ==============================================================
# 2) Crear quintiles (5 grupos) por score descendente
# ==============================================================
df_final["quintil"] = pd.qcut(df_final["score"], 5, labels=["Q1","Q2","Q3","Q4","Q5"][::-1])
# Q1 = más riesgoso (top 20%), Q5 = menos riesgoso

# ==============================================================
# 3) Calcular métricas por quintil usando las variables del modelo
# ==============================================================
resumen = df_final.groupby("quintil").agg(
    clientes=("key_value", "nunique"),
    monto_abonos_3m_max=("trx_monto_abonos_3m_max", "mean"),  # Máximo abono 3m
    pasivo_soles_prom=("pasivo_soles", "mean"),  # Promedio pasivos en soles
    q_meses_ingresos_0_prom=("q_meses_ingresos_0", "mean"),  # Promedio meses sin ingresos
    abonos_6m_efectivo_prom=("trx_monto_abonos_6m_efectivo", "mean"),  # Promedio abonos en efectivo 6m
    q_meses_egresos_0_prom=("cnt_meses_sinegresos_12m", "mean"),  # Promedio meses sin egresos
    antiguedad_prom=("num_antiguedad", "mean"),  # Promedio años de relación
  #  dif_abonos_cargos_6m_prom=("dif_q_abonos_cargos_efectivo_6m", "mean"),  # Dif. abonos-cargos 6m
    cargos_6m_efectivo_prom=("trx_monto_cargos_6m_efectivo", "mean"),  # Promedio cargos en efectivo 6m
    abonos_3m_total_prom=("trx_q_abonos_promedio_3m_total", "mean"),  # Promedio abonos totales 3m
    ratio_abonos_9m_efectivo=("trx_q_abonos_ratio_9m_efectivo_total", "mean"),  # Ratio abonos 9m
    edad_constitucion_prom=("edad_constitucion", "mean"),  # Promedio años de constitución
    riesgo_real_pct=("target", "mean"),  # % de clientes con evento real
    riesgo_real_abs=("target", "sum"),
    pct_lima=("desc_departamento", lambda x: (x=="LIMA").mean() * 100),
    pct_trujillo=("desc_departamento", lambda x: (x=="LA LIBERTAD").mean() * 100),
    pct_san_roman=("desc_departamento", lambda x: (x=="PUNO").mean() * 100),
    # Alertas con riesgo (basado en tipo_alerta_n2, asumimos que está en df_final)
    pct_alertas_riesgo=("tipo_alerta_n2", lambda x: (x.str.contains("ALTO|Riesgo", case=False, na=False)).mean() * 100)
).round(2)

# Ordenar de mayor a menor riesgo
resumen = resumen.loc[["Q1", "Q2", "Q3", "Q4", "Q5"]].copy()
resumen["clientes_acum"] = resumen["clientes"].cumsum()
total_clientes = resumen["clientes"].sum()
total_riesgo = resumen["riesgo_real_abs"].sum()

resumen["pct_clientes"] = (resumen["clientes"] / total_clientes * 100).round(1)
resumen["pct_riesgo_acum"] = (resumen["riesgo_real_abs"].cumsum() / total_riesgo * 100).round(1)

# ==============================================================
# 4) Imprimir los valores para el HTML
# ==============================================================
print("VALORES PARA EL HTML (cópialos directamente):")
print("="*60)

s1 = resumen.loc["Q1"]
s2 = resumen.loc["Q2"]
s3 = resumen.loc["Q3"]
s4_s5 = resumen.loc[["Q4", "Q5"]].sum()

# Formatear montos con formato peruano (S/)
def format_soles(valor):
    return f"S/ {valor:,.1f}".replace(",", "X").replace(".", ",").replace("X", ".")

print(f"S1_clientes = {int(s1.clientes)}")
print(f"S1_monto_abonos_3m_max = {format_soles(s1.monto_abonos_3m_max)}")
print(f"S1_pasivo_soles = {format_soles(s1.pasivo_soles_prom)}")
print(f"S1_q_meses_ingresos_0 = {s1.q_meses_ingresos_0_prom:.1f}")
print(f"S1_abonos_6m_efectivo = {format_soles(s1.abonos_6m_efectivo_prom)}")
print(f"S1_q_meses_egresos_0 = {s1.q_meses_egresos_0_prom:.1f}")
print(f"S1_antiguedad = {s1.antiguedad_prom:.1f}")

print(f"S1_cargos_6m_efectivo = {format_soles(s1.cargos_6m_efectivo_prom)}")
print(f"S1_abonos_3m_total = {s1.abonos_3m_total_prom:.1f}")
print(f"S1_ratio_abonos_9m = {s1.ratio_abonos_9m_efectivo:.2f}")
print(f"S1_edad_constitucion = {s1.edad_constitucion_prom:.1f}")
print(f"S1_riesgo_real = {s1.riesgo_real_pct:.2f}")
print(f"S1_lima = {s1.pct_lima:.1f}")
print(f"S1_trujillo = {s1.pct_trujillo:.1f}")
print(f"S1_san_roman = {s1.pct_san_roman:.1f}")
print(f"S1_alertas_riesgo = {s1.pct_alertas_riesgo:.1f}")

print(f"\nS2_clientes = {int(s2.clientes)}")
print(f"S2_monto_abonos_3m_max = {format_soles(s2.monto_abonos_3m_max)}")
print(f"S2_pasivo_soles = {format_soles(s2.pasivo_soles_prom)}")
print(f"S2_q_meses_ingresos_0 = {s2.q_meses_ingresos_0_prom:.1f}")
print(f"S2_abonos_6m_efectivo = {format_soles(s2.abonos_6m_efectivo_prom)}")
print(f"S2_q_meses_egresos_0 = {s2.q_meses_egresos_0_prom:.1f}")
print(f"S2_antiguedad = {s2.antiguedad_prom:.1f}")

print(f"S2_cargos_6m_efectivo = {format_soles(s2.cargos_6m_efectivo_prom)}")
print(f"S2_abonos_3m_total = {s2.abonos_3m_total_prom:.1f}")
print(f"S2_ratio_abonos_9m = {s2.ratio_abonos_9m_efectivo:.2f}")
print(f"S2_edad_constitucion = {s2.edad_constitucion_prom:.1f}")
print(f"S2_lima = {s2.pct_lima:.1f}")
print(f"S2_trujillo = {s2.pct_trujillo:.1f}")
print(f"S2_san_roman = {s2.pct_san_roman:.1f}")
print(f"S2_alertas_riesgo = {s2.pct_alertas_riesgo:.1f}")

print(f"\nS3_clientes = {int(s3.clientes)}")
print(f"S3_monto_abonos_3m_max = {format_soles(s3.monto_abonos_3m_max)}")
print(f"S3_pasivo_soles = {format_soles(s3.pasivo_soles_prom)}")
print(f"S3_q_meses_ingresos_0 = {s3.q_meses_ingresos_0_prom:.1f}")
print(f"S3_abonos_6m_efectivo = {format_soles(s3.abonos_6m_efectivo_prom)}")
print(f"S3_q_meses_egresos_0 = {s3.q_meses_egresos_0_prom:.1f}")
print(f"S3_antiguedad = {s3.antiguedad_prom:.1f}")

print(f"S3_cargos_6m_efectivo = {format_soles(s3.cargos_6m_efectivo_prom)}")
print(f"S3_abonos_3m_total = {s3.abonos_3m_total_prom:.1f}")
print(f"S3_ratio_abonos_9m = {s3.ratio_abonos_9m_efectivo:.2f}")
print(f"S3_edad_constitucion = {s3.edad_constitucion_prom:.1f}")

print(f"\nS4S5_clientes = {int(s4_s5.clientes)}")
print(f"S4S5_monto_abonos_3m_max = {format_soles(s4_s5.monto_abonos_3m_max)}")
print(f"S4S5_pasivo_soles = {format_soles(s4_s5.pasivo_soles_prom)}")
print(f"S4S5_q_meses_ingresos_0 = {s4_s5.q_meses_ingresos_0_prom:.1f}")
print(f"S4S5_abonos_6m_efectivo = {format_soles(s4_s5.abonos_6m_efectivo_prom)}")
print(f"S4S5_q_meses_egresos_0 = {s4_s5.q_meses_egresos_0_prom:.1f}")
print(f"S4S5_antiguedad = {s4_s5.antiguedad_prom:.1f}")

print(f"S4S5_cargos_6m_efectivo = {format_soles(s4_s5.cargos_6m_efectivo_prom)}")
print(f"S4S5_abonos_3m_total = {s4_s5.abonos_3m_total_prom:.1f}")
print(f"S4S5_ratio_abonos_9m = {s4_s5.ratio_abonos_9m_efectivo:.2f}")
print(f"S4S5_edad_constitucion = {s4_s5.edad_constitucion_prom:.1f}")

print(f"\nCONCENTRACIÓN:")
print(f"pct_s1_s2 = {resumen.loc[['Q1','Q2'],'pct_clientes'].sum():.1f}")
print(f"pct_riesgo_s1_s2 = {resumen.loc[['Q1','Q2'],'pct_riesgo_acum'].iloc[-1]:.1f}")
print(f"pct_alertas_s1_s2 = {(resumen.loc[['Q1','Q2'],'pct_alertas_riesgo'].mean()):.1f}")

VALORES PARA EL HTML (cópialos directamente):
S1_clientes = 109
S1_monto_abonos_3m_max = S/ 3.526.618,9
S1_pasivo_soles = S/ 188.617,1
S1_q_meses_ingresos_0 = 6.8
S1_abonos_6m_efectivo = S/ 1.172.868,7
S1_q_meses_egresos_0 = 6.0
S1_antiguedad = 0.8
S1_cargos_6m_efectivo = S/ 1.022.371,0
S1_abonos_3m_total = 39.3
S1_ratio_abonos_9m = 0.46
S1_edad_constitucion = 1.2
S1_riesgo_real = 0.63
S1_lima = 40.9
S1_trujillo = 13.9
S1_san_roman = 16.5
S1_alertas_riesgo = 0.0

S2_clientes = 110
S2_monto_abonos_3m_max = S/ 3.745.166,8
S2_pasivo_soles = S/ 1.083.736,2
S2_q_meses_ingresos_0 = 3.7
S2_abonos_6m_efectivo = S/ 1.531.184,0
S2_q_meses_egresos_0 = 3.3
S2_antiguedad = 2.6
S2_cargos_6m_efectivo = S/ 798.340,6
S2_abonos_3m_total = 52.8
S2_ratio_abonos_9m = 0.27
S2_edad_constitucion = 6.2
S2_lima = 56.1
S2_trujillo = 9.7
S2_san_roman = 7.9
S2_alertas_riesgo = 0.0

S3_clientes = 115
S3_monto_abonos_3m_max = S/ 3.935.155,4
S3_pasivo_soles = S/ 659.634,4
S3_q_meses_ingresos_0 = 2.4
S3_abonos_6m_efec

In [113]:
import pandas as pd
import numpy as np

# ==============================================================
# 1) QUINTILES CORRECTOS: S1 = MÁS RIESGO (score más alto)
# ==============================================================
# Forzamos 5 grupos exactamente iguales en cantidad de clientes
df_final["quintil"] = pd.qcut(-df_final["score"], 
                              q=5, 
                              labels=["S1", "S2", "S3", "S4", "S5"])

# ==============================================================
# 2) RESUMEN GENERAL
# ==============================================================
resumen = df_final.groupby("quintil").agg(
    clientes=("num_documento", "nunique"),
    eventos=("target", "sum"),
    event_rate=("target", "mean"),
    score_prom=("score", "mean"),
    
    abonos_3m_max_prom=("trx_monto_abonos_3m_max", "mean"),
    pasivo_soles_prom=("pasivo_soles", "mean"),
    q_meses_sin_ingresos=("q_meses_ingresos_0", "mean"),
    antiguedad_prom=("antiguedad", "mean"),
    edad_constitucion_prom=("edad_constitucion", "mean"),
    
    pct_lima=("departamento", lambda x: (x == "LIMA").mean() * 100),
    pct_trujillo=("departamento", lambda x: (x == "LA LIBERTAD").mean() * 100),
).round(3)

# Métricas globales
total_clientes = resumen["clientes"].sum()
total_eventos = resumen["eventos"].sum()

resumen["pct_clientes"] = (resumen["clientes"] / total_clientes * 100).round(1)
resumen["pct_riesgo_acum"] = (resumen["eventos"].cumsum() / total_eventos * 100).round(1)
tasa_global = df_final["target"].mean()
resumen["lift"] = (resumen["event_rate"] / tasa_global).round(2)

# ==============================================================
# 3) TIPOS DE ALERTA
# ==============================================================
if "tipo_alerta_n2" not in df_final.columns:
    df_final = df_final.merge(df_4[["num_documento", "mes_base", "tipo_alerta_n2"]],
                              on=["num_documento", "mes_base"], how="left")

df_final["alerta"] = df_final["tipo_alerta_n2"].fillna("SIN ALERTA").str.strip().str.upper()

alertas = (df_final.groupby("quintil")["alerta"]
           .value_counts(normalize=True)
           .mul(100)
           .round(1)
           .unstack(fill_value=0))

# ==============================================================
# 4) FUNCIÓN PARA FORMATO S/
# ==============================================================
def soles(x):
    return f"S/ {x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")

# ==============================================================
# 5) IMPRESIÓN BONITA Y LISTA PARA LA DIAPOSITIVA
# ==============================================================
print("\n" + "="*100)
print("RESULTADOS POR QUINTIL – PLAFT 2025 – LISTO PARA PRESENTACIÓN")
print("="*100)

for seg in ["S1", "S2", "S3"]:
    s = resumen.loc[seg]
    print(f"\n{seg} → {int(s.clientes)} clientes | {int(s.eventos)} casos reales | {s.event_rate*100:.1f}% event rate | Lift {s.lift}x")
    print(f"   Score promedio: {s.score_prom:.4f}")
    print(f"   Abonos máx 3m: {soles(s.abonos_3m_max_prom)}")
    print(f"   Pasivos: {soles(s.pasivo_soles_prom)}")
    print(f"   Antigüedad: {s.antiguedad_prom:.1f} años | Edad constitución: {s.edad_constitucion_prom:.1f} años")
    print(f"   Meses sin ingresos: {s.q_meses_sin_ingresos:.1f}")
    print(f"   % Lima: {s.pct_lima:.1f}% | % Trujillo: {s.pct_trujillo:.1f}%")
    print(f"   Alertas → ", end="")
    print(" | ".join([f"{col}: {alertas.loc[seg,col]}%" for col in alertas.columns if alertas.loc[seg,col] > 0]))

# S4+S5 juntos
s4s5 = resumen.loc[["S4","S5"]].sum()
s4s5.event_rate = s4s5.eventos / s4s5.clientes
s4s5.lift = round(s4s5.event_rate / tasa_global, 2)

print(f"\nS4+S5 → {int(s4s5.clientes)} clientes | {int(s4s5.eventos)} casos reales | {s4s5.event_rate*100:.1f}% event rate | Lift {s4s5.lift}x")
print(f"   Casi todo el volumen actual de alertas está aquí… pero con riesgo mínimo")

print(f"\nCONCENTRACIÓN FINAL:")
print(f"• S1+S2 = {resumen.loc[['S1','S2'],'pct_clientes'].sum():.1f}% de los clientes")
print(f"• Capturan el {resumen.loc[['S1','S2'],'pct_riesgo_acum'].iloc[-1]:.1f}% del riesgo real")
print(f"• Lift promedio S1+S2 ≈ {resumen.loc[['S1','S2'],'lift'].mean():.2f}x")

print("\n" + "="*100)
print("¡PEGA ESTA SALIDA EN EL CHAT Y EN 2 MINUTOS TE DOY EL HTML FINAL MÁS BRUTAL QUE HAYAS TENIDO!")
print("="*100)


RESULTADOS POR QUINTIL – PLAFT 2025 – LISTO PARA PRESENTACIÓN

S1 → 107 clientes | 79 casos reales | 68.7% event rate | Lift 3.1x
   Score promedio: 0.7770
   Abonos máx 3m: S/ 4.224.989
   Pasivos: S/ 281.744
   Antigüedad: 0.7 años | Edad constitución: 0.9 años
   Meses sin ingresos: 6.8
   % Lima: 43.5% | % Trujillo: 13.0%
   Alertas → AUTOMATICA: 33.9% | MANUAL: 46.1% | SEMI AUTOMATICA: 20.0%

S2 → 112 clientes | 30 casos reales | 26.3% event rate | Lift 1.19x
   Score promedio: 0.0970
   Abonos máx 3m: S/ 3.362.893
   Pasivos: S/ 1.168.848
   Antigüedad: 2.6 años | Edad constitución: 7.1 años
   Meses sin ingresos: 4.2
   % Lima: 51.8% | % Trujillo: 10.5%
   Alertas → AUTOMATICA: 66.7% | MANUAL: 21.1% | SEMI AUTOMATICA: 12.3%

S3 → 115 clientes | 13 casos reales | 11.3% event rate | Lift 0.51x
   Score promedio: 0.0100
   Abonos máx 3m: S/ 4.777.637
   Pasivos: S/ 829.478
   Antigüedad: 7.4 años | Edad constitución: 8.7 años
   Meses sin ingresos: 1.7
   % Lima: 60.0% | % Trujill

In [116]:
import pandas as pd
import numpy as np

# ==============================================================
# 1) CREAR QUINTILES CORRECTOS: S1 = MÁS ALTO SCORE = MÁS RIESGO
# ==============================================================
df_final["quintil"] = pd.qcut(-df_final["score"], 5, labels=["S1", "S2", "S3", "S4", "S5"])

# ==============================================================
# 2) TODAS LAS VARIABLES DEL MODELO (promedios)
# ==============================================================
resumen = df_final.groupby("quintil").agg(
    clientes=("num_documento", "nunique"),
    eventos=("target", "sum"),
    event_rate=("target", "mean"),
    score_prom=("score", "mean"),
    
    # Variables del modelo (promedios)
    abonos_3m_max_prom=("trx_monto_abonos_3m_max", "mean"),
    pasivo_soles_prom=("pasivo_soles", "mean"),
    q_meses_sin_ingresos=("q_meses_ingresos_0", "mean"),
    abonos_6m_efectivo_prom=("trx_monto_abonos_6m_efectivo", "mean"),
    q_meses_sin_egresos=("q_meses_egresos_0", "mean"),
    antiguedad_prom=("antiguedad", "mean"),
    cargos_6m_efectivo_prom=("trx_monto_cargos_6m_efectivo", "mean"),
    abonos_promedio_3m_total=("trx_q_abonos_promedio_3m_total", "mean"),
    ratio_abonos_9m_efectivo=("trx_q_abonos_ratio_9m_efectivo_total", "mean"),
    edad_constitucion_prom=("edad_constitucion", "mean"),
    ratio_abonos_1m_efectivo=("trx_q_abonos_ratio_1m_efectivo_total", "mean"),
    ratio_cargos_1m_efectivo=("trx_monto_cargos_ratio_1m_efectivo_total", "mean"),
    cargos_promedio_3m_total=("trx_monto_cargos_promedio_3m_total", "mean"),
    
    # Geografía
    pct_lima=("departamento", lambda x: (x == "LIMA").mean() * 100),
    pct_trujillo=("departamento", lambda x: (x == "LA LIBERTAD").mean() * 100),
    pct_san_roman=("departamento", lambda x: (x == "PUNO").mean() * 100)
).round(3)

# Métricas globales
total_clientes = resumen["clientes"].sum()
total_eventos = resumen["eventos"].sum()
resumen["pct_clientes"] = (resumen["clientes"] / total_clientes * 100).round(1)
resumen["pct_riesgo_acum"] = (resumen["eventos"].cumsum() / total_eventos * 100).round(1)
resumen["lift"] = (resumen["event_rate"] / df_final["target"].mean()).round(2)

# ==============================================================
# 3) TIPOS DE ALERTA POR SEGMENTO
# ==============================================================
if "tipo_alerta_n2" not in df_final.columns:
    df_final = df_final.merge(df_4[["num_documento", "mes_base", "tipo_alerta_n2"]], 
                              on=["num_documento", "mes_base"], how="left")

df_final["alerta"] = df_final["tipo_alerta_n2"].fillna("SIN ALERTA").str.strip().str.upper()

alertas = (df_final.groupby("quintil")["alerta"]
           .value_counts(normalize=True)
           .mul(100)
           .round(1)
           .unstack(fill_value=0))

# ==============================================================
# 4) IMPRIMIR TODO (S1, S2, S3, S4+S5) - LISTO PARA HTML
# ==============================================================
print("\n" + "="*90)
print("RESULTADOS COMPLETOS POR SEGMENTO - LISTO PARA DIAPOSITIVA PLAFT")
print("="*90)

def soles(x):
    return f"S/ {x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")

seg_names = ["S1", "S2", "S3"]
for seg in seg_names + ["S4+S5"]:
    if seg != "S4+S5":
        s = resumen.loc[seg]
        print(f"\n--- {seg} (Top riesgo) ---")
    else:
        s = resumen.loc[["S4","S5"]].sum()
        s.event_rate = (s.eventos / s.clientes)
        s.lift = (s.event_rate / df_final["target"].mean()).round(2)
        print(f"\n--- {seg} (Bajo riesgo) ---")
    
    print(f"Clientes: {int(s.clientes)} | Eventos: {int(s.eventos)} | Event Rate: {s.event_rate*100:.1f}% | Lift: {s.lift:.2f}x")
    print(f"Score promedio: {s.score_prom:.4f}")
    print(f"Abonos máx 3m (prom): {soles(s.abonos_3m_max_prom)}")
    print(f"Pasivos soles (prom): {soles(s.pasivo_soles_prom)}")
    print(f"Antigüedad (prom): {s.antiguedad_prom:.1f} años")
 
    print(f"Cargos efectivo 6m (prom): {soles(s.abonos_6m_efectivo_prom)}")
    print(f"Meses sin ingresos (prom): {s.q_meses_sin_ingresos:.1f}")
    print(f"Ratio abonos efectivo 9m: {s.ratio_abonos_9m_efectivo:.3f}")
    print(f"Edad constitución: {s.edad_constitucion_prom:.1f} años")
    print(f"% Lima: {s.pct_lima:.1f}%")
    
    # Alertas
    if seg != "S4+S5":
        print(f"TIPOS DE ALERTA {seg}:")
        for col in alertas.columns:
            if seg in alertas.index and alertas.loc[seg, col] > 0:
                print(f"   → {col}: {alertas.loc[seg, col]}%")
    else:
        print(f"TIPOS DE ALERTA S4+S5:")
        s4s5_alert = df_final[df_final["quintil"].isin(["S4","S5"])]["alerta"].value_counts(normalize=True).mul(100).round(1)
        for tipo, pct in s4s5_alert.items():
            print(f"   → {tipo}: {pct}%")

print(f"\nCONCENTRACIÓN FINAL:")
print(f"S1+S2 clientes: {resumen.loc[['S1','S2'],'pct_clientes'].sum():.1f}%")
print(f"S1+S2 riesgo capturado: {resumen.loc[['S1','S2'],'pct_riesgo_acum'].iloc[-1]:.1f}%")

print("\n" + "="*90)
print("¡EJECUTA Y PÉGAME TODA ESTA SALIDA PARA ARMAR EL HTML DEFINITIVO!")
print("="*90)


RESULTADOS COMPLETOS POR SEGMENTO - LISTO PARA DIAPOSITIVA PLAFT

--- S1 (Top riesgo) ---
Clientes: 107 | Eventos: 79 | Event Rate: 68.7% | Lift: 3.10x
Score promedio: 0.7770
Abonos máx 3m (prom): S/ 4.224.989
Pasivos soles (prom): S/ 281.744
Antigüedad (prom): 0.7 años
Cargos efectivo 6m (prom): S/ 1.244.912
Meses sin ingresos (prom): 6.8
Ratio abonos efectivo 9m: 0.484
Edad constitución: 0.9 años
% Lima: 43.5%
TIPOS DE ALERTA S1:
   → AUTOMATICA: 33.9%
   → MANUAL: 46.1%
   → SEMI AUTOMATICA: 20.0%

--- S2 (Top riesgo) ---
Clientes: 112 | Eventos: 30 | Event Rate: 26.3% | Lift: 1.19x
Score promedio: 0.0970
Abonos máx 3m (prom): S/ 3.362.893
Pasivos soles (prom): S/ 1.168.848
Antigüedad (prom): 2.6 años
Cargos efectivo 6m (prom): S/ 808.893
Meses sin ingresos (prom): 4.2
Ratio abonos efectivo 9m: 0.214
Edad constitución: 7.1 años
% Lima: 51.8%
TIPOS DE ALERTA S2:
   → AUTOMATICA: 66.7%
   → MANUAL: 21.1%
   → SEMI AUTOMATICA: 12.3%

--- S3 (Top riesgo) ---
Clientes: 115 | Eventos: 13

In [ ]:
import pandas as pd
import numpy as np

# ==============================================================
# 1) CREAR QUINTILES CORRECTOS: S1 = MÁS ALTO SCORE = MÁS RIESGO
# ==============================================================
# Esta es la línea mágica que arregla todo en tu caso:
df_final["quintil"] = pd.qcut(-df_final["score"], 5, labels=["S1", "S2", "S3", "S4", "S5"])
# El signo "-" invierte el orden → los scores más altos van a S1

# ==============================================================
# 2) MÉTRICAS CLAVE POR SEGMENTO
# ==============================================================
resumen = df_final.groupby("quintil").agg(
    clientes=("num_documento", "nunique"),
    eventos=("target", "sum"),
    event_rate=("target", "mean"),
    score_prom=("score", "mean"),
    monto_max_abono_3m=("trx_monto_abonos_3m_max", "mean"),
    pasivo_soles=("pasivo_soles", "mean"),
    antiguedad=("antiguedad", "mean"),
    dif_abonos_cargos=("dif_q_abonos_cargos_efectivo_6m", "mean"),
    cargos_efectivo_6m=("trx_monto_cargos_6m_efectivo", "mean"),
    pct_lima=("departamento", lambda x: (x == "LIMA").mean() * 100)
).round(3)

total_clientes = resumen["clientes"].sum()
total_eventos = resumen["eventos"].sum()
resumen["pct_clientes"] = (resumen["clientes"] / total_clientes * 100).round(1)
resumen["pct_riesgo_acum"] = (resumen["eventos"].cumsum() / total_eventos * 100).round(1)
resumen["lift"] = (resumen["event_rate"] / df_final["target"].mean()).round(2)

# ==============================================================
# 3) TIPOS DE ALERTA POR SEGMENTO (AUTOMÁTICA, MANUAL, etc.)
# ==============================================================
if "tipo_alerta_n2" not in df_final.columns:
    df_final = df_final.merge(
        df_4[["num_documento", "mes_base", "tipo_alerta_n2"]],
        on=["num_documento", "mes_base"],
        how="left"
    )

df_final["alerta"] = df_final["tipo_alerta_n2"].fillna("SIN ALERTA").str.strip().str.upper()

alertas = (
    df_final.groupby("quintil")["alerta"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .unstack(fill_value=0)
)

# ==============================================================
# 4) IMPRIMIR TODO LISTO PARA EL HTML
# ==============================================================
print("\n" + "="*80)
print("SALIDA FINAL PARA TU DIAPOSITIVA PLAFT (S1 = MÁS RIESGO)")
print("="*80)

s1 = resumen.loc["S1"]
s2 = resumen.loc["S2"]
s3 = resumen.loc["S3"]
s4s5 = resumen.loc[["S4", "S5"]].sum()

def soles(x):
    return f"S/ {x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")

print(f"S1_clientes = {int(s1.clientes)}")
print(f"S1_eventos = {int(s1.eventos)}")
print(f"S1_event_rate = {s1.event_rate*100:.1f}%")
print(f"S1_lift = {s1.lift:.2f}x")
print(f"S1_monto_max_3m = {soles(s1.monto_max_abono_3m)}")
print(f"S1_pasivo_prom = {soles(s1.pasivo_soles)}")
print(f"S1_antiguedad = {s1.antiguedad:.1f} años")
print(f"S1_dif_abonos_cargos = {s1.dif_abonos_cargos:+.1f}")
print(f"S1_pct_lima = {s1.pct_lima:.1f}%")

print(f"\nS2_clientes = {int(s2.clientes)}")
print(f"S2_eventos = {int(s2.eventos)}")
print(f"S2_event_rate = {s2.event_rate*100:.1f}%")
print(f"S2_lift = {s2.lift:.2f}x")

print(f"\nCONCENTRACIÓN S1+S2:")
print(f"Clientes_S1S2 = {int(s1.clientes + s2.clientes)} ({resumen.loc[['S1','S2'],'pct_clientes'].sum():.1f}%)")
print(f"Riesgo_capturado_S1S2 = {resumen.loc[['S1','S2'],'pct_riesgo_acum'].iloc[-1]:.1f}%")

print(f"\nTIPOS DE ALERTA POR SEGMENTO")
print("-"*60)
for seg, nombre in [("S1","S1"), ("S2","S2"), ("S3","S3")]:
    print(f"\n→ {nombre}:")
    if seg in alertas.index:
        for tipo in alertas.columns:
            pct = alertas.loc[seg, tipo]
            if pct > 0:
                print(f"   • {tipo}: {pct}%")
    else:
        print("   • Sin datos")

print(f"\n→ S4+S5:")
alertas_s4s5 = df_final[df_final["quintil"].isin(["S4","S5"])]["alerta"].value_counts(normalize=True).mul(100).round(1)
for tipo, pct in alertas_s4s5.items():
    print(f"   • {tipo}: {pct}%")

print("\n" + "="*80)
print("¡EJECUTA ESTO Y PÉGAME LA SALIDA COMPLETA!")
print("="*80)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score

# -------------------------------
# 1️⃣ Matriz con umbral 0.3
# -------------------------------
umbral_default = 0.35
y_pred_default = (y_pred_prob >= umbral_default).astype(int)

# Matriz de confusión
cm_default = confusion_matrix(y_test, y_pred_default)
disp_default = ConfusionMatrixDisplay(confusion_matrix=cm_default)
print(f"\n📌 Matriz de Confusión (umbral {umbral_default})")
disp_default.plot()

# -------------------------------
# 2️⃣ Precision y Recall
# -------------------------------
precision = precision_score(y_test, y_pred_default)
recall = recall_score(y_test, y_pred_default)

print(f"\n🎯 Precision: {precision:.4f}")
print(f"📈 Recall: {recall:.4f}")

# Base completa con 2 meses test y 2 meses de val

In [ ]:
import pandas as pd
from sklearn.utils import resample

# ============================================================
# 1️⃣ DEFINIR SPLIT DE DATOS
# ============================================================

df_train_raw = df_3[df_3["mes_base"] < 202506].copy()
df_val_raw   = df_3[df_3["mes_base"].isin([202506, 202507])].copy()
df_test_raw  = df_3[df_3["mes_base"].isin([202508, 202509])].copy()


# ============================================================
# 2️⃣ EN TRAIN: separar con alerta y sin alerta
# ============================================================

df_train_con_alerta  = df_train_raw[df_train_raw["tipo_alerta_n2"] != "0"]
df_train_sin_alerta  = df_train_raw[df_train_raw["tipo_alerta_n2"] == "0"]


# ============================================================
# 3️⃣ Separar clases en TRAIN
# ============================================================

df_train_1 = df_train_con_alerta[df_train_con_alerta["target"] == 1]
df_train_0 = df_train_con_alerta[df_train_con_alerta["target"] == 0]


# ============================================================
# 4️⃣ Balanceo:
#        Queremos 0.5% → target=1
#        Queremos 99.5% → target=0
# ============================================================

n_pos = len(df_train_1)
n_neg_deseado = int(n_pos * (99.5 / 0.5))  # relación 199:1

if n_neg_deseado <= len(df_train_0):
    df_train_0_bal = resample(df_train_0, replace=False, n_samples=n_neg_deseado, random_state=42)
else:
    df_train_0_bal = resample(df_train_0, replace=True, n_samples=n_neg_deseado, random_state=42)

df_train_1_bal = df_train_1


# ============================================================
# 5️⃣ Agregar 0.5% de casos SIN alerta
# ============================================================

df_sin_alerta_sample = df_train_sin_alerta.sample(frac=0.005, random_state=42)


# ============================================================
# 6️⃣ Construir TRAIN final
# ============================================================

df_train = pd.concat([
    df_train_0_bal,
    df_train_1_bal,
    df_sin_alerta_sample
], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)


# ============================================================
# 7️⃣ VALIDACIÓN → se deja intacta
# ============================================================

df_val = df_val_raw.copy()


# ============================================================
# 8️⃣ TEST → meses 202508 y 202509 completos
# ============================================================

df_test = df_test_raw.copy()


# ============================================================
# 9️⃣ SALIDAS
# ============================================================

print("📌 TRAIN (target):")
print(df_train["target"].value_counts(normalize=True))

print("\n📌 VAL (target):")
print(df_val["target"].value_counts(normalize=True))

print("\n📌 TEST (target):")
print(df_test["target"].value_counts(normalize=True))

print("\nTamaños finales:")
print("TRAIN:", df_train.shape)
print("VAL:  ", df_val.shape)
print("TEST: ", df_test.shape)


In [ ]:
df_5 = pd.concat([df_train,df_val, df_test], axis=0).reset_index(drop=True)

In [ ]:
cols_a_eliminar = ["fecha_constitucion"]
df_5 = df_5.drop(columns=cols_a_eliminar, errors="ignore")
df_5[['mes_base', 'target']].value_counts()
# Convertir explícitamente la columna problemática a string
df_5['tipo_alerta_n2'] = df_5['tipo_alerta_n2'].astype(str)

In [ ]:
df_5[['mes_base', 'target']].value_counts()

In [ ]:
df_5.to_parquet(
    's3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/DATA_INFERENCIA/data_pn_total_train_val_test.parquet',
    index=False
)

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# ============================================================
# 1️⃣ Separar TRAIN / VAL / TEST por mes_base
# ============================================================



# Guardamos para reconstruir después
df_all = pd.concat([df_train, df_val, df_test], axis=0).copy()

from sklearn.preprocessing import LabelEncoder
import pandas as pd

# ============================================================
# 1️⃣ Codificar categorías en df_all (NO hacemos splits aún)
# ============================================================

df_encoded = df_all.copy()

categoricas = [
    "flag_casos_hist",
    "flag_variacion_abono_monto_total_5m_1m",
    "flag_variacion_efect_cargos_monto_5m_1m",
    "provincia",
    "departamento",
    "ubigeo_cd",
    "sectorista_id",
    "ciiu_v4"
]

encoders = {}

for c in categoricas:
    le = LabelEncoder()
    df_encoded[c] = le.fit_transform(df_encoded[c].astype(str))
    encoders[c] = le

# ============================================================
# 2️⃣ Ahora sí reconstruimos los splits con mes_base original
# ============================================================

df_train = df_encoded[df_encoded["mes_base"] <= 202505].copy()       # Train
df_val   = df_encoded[df_encoded["mes_base"].isin([202506, 202507])].copy()  # Val
df_test  = df_encoded[df_encoded["mes_base"].isin([202508, 202509])].copy()  # Test

print("Train:", df_train.shape)
print("Val:  ", df_val.shape)
print("Test: ", df_test.shape)



In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from scipy.stats import randint, uniform

# =====================================================
# 1️⃣ Separar features / target
# =====================================================

target = "target"

X_train = df_train.drop(columns=[target, "mes_base", "tipo_alerta_n2", "num_documento"], errors="ignore")
y_train = df_train[target]

X_val   = df_val.drop(columns=[target, "mes_base", "tipo_alerta_n2", "num_documento"], errors="ignore")
y_val   = df_val[target]

X_test  = df_test.drop(columns=[target, "mes_base", "tipo_alerta_n2", "num_documento"], errors="ignore")
y_test  = df_test[target]

# =====================================================
# 2️⃣ Todas las variables son numéricas (LabelEncoded)
# =====================================================

pipeline_xgb = Pipeline(steps=[
    ("xgb", XGBClassifier(
        eval_metric="logloss",
        tree_method="hist",
        random_state=42
    ))
])

# =====================================================
# 3️⃣ HPO
# =====================================================

param_dist = {
    "xgb__max_depth": randint(3, 8),
    "xgb__learning_rate": uniform(0.01, 0.15),
    "xgb__subsample": uniform(0.6, 0.4),
    "xgb__colsample_bytree": uniform(0.6, 0.4),
    "xgb__gamma": uniform(0, 5),
    "xgb__min_child_weight": randint(1, 10),
    "xgb__n_estimators": randint(200, 800)
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

search_xgb = RandomizedSearchCV(
    estimator=pipeline_xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring="precision",
    cv=cv,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

search_xgb.fit(X_train, y_train)

print("\n🏆 Mejores hiperparámetros:")
print(search_xgb.best_params_)

# =====================================================
# 4️⃣ Evaluación en TEST
# =====================================================

y_pred_prob = search_xgb.predict_proba(X_test)[:,1]

def ks_score(y_true, y_score):
    df = pd.DataFrame({"y": y_true, "score": y_score})
    df = df.sort_values("score", ascending=False)
    df["cum_event"] = (df["y"]==1).cumsum() / df["y"].sum()
    df["cum_nonevent"] = (df["y"]==0).cumsum() / (len(df) - df["y"].sum())
    return (df["cum_event"] - df["cum_nonevent"]).max()

auc  = roc_auc_score(y_test, y_pred_prob)
gini = 2*auc - 1
ks   = ks_score(y_test, y_pred_prob)

print("\n📊 MÉTRICAS EN TEST:")
print(f"AUC   = {auc:.4f}")
print(f"Gini  = {gini:.4f}")
print(f"KS    = {ks:.4f}")


In [ ]:
import pandas as pd
import numpy as np

df_val["score"]  = search_xgb.predict_proba(X_val)[:,1]
df_test["score"] = search_xgb.predict_proba(X_test)[:,1]

df_eval = pd.concat([df_val, df_test], axis=0).reset_index(drop=True)

print("Meses incluidos:", df_eval["mes_base"].unique())

# ======================================================================
# 1️⃣ Definir percentiles acumulativos
# ======================================================================
percentiles = [0.01, 0.05, 0.10, 0.50, 1.00]
labels = ["Top 1%", "Top 5%", "Top 10%", "Top 50%", "Total"]

resultados = []

# ======================================================================
# 2️⃣ Loop por mes
# ======================================================================
for mes in sorted(df_eval["mes_base"].unique()):
    
    df_mes = df_eval[df_eval["mes_base"] == mes].copy()
    
    # Ordenar descendentemente por score
    df_mes = df_mes.sort_values("score", ascending=False).reset_index(drop=True)
    
    # Ranking en percentil
    df_mes["percentile"] = (df_mes.index + 1) / len(df_mes)

    tablas = []
    
    for p, label in zip(percentiles, labels):
        df_cut = df_mes[df_mes["percentile"] <= p]

        tabla = {
            "grupo": label,
            "count": len(df_cut),
            "events": df_cut["target"].sum(),
            "precision": df_cut["target"].mean(),
            "recall": df_cut["target"].sum() / df_mes["target"].sum(),
            "score_mean": df_cut["score"].mean(),
            "mes_base": mes
        }
        tablas.append(tabla)
    
    resultados.extend(tablas)

# ======================================================================
# 3️⃣ Resultado final
# ======================================================================
df_result_final = pd.DataFrame(resultados)

print("\n📊 EFECTIVIDAD ACUMULADA POR GRUPOS (1%,5%,10%,50%,100%)")
print(df_result_final)



In [ ]:
import pandas as pd

# ===============================================================
# CONFIG: cantidad de grupos
# ===============================================================
N_GROUPS = 500      # 500 grupos iguales
TOP_SHOW = 5        # mostrar solo los primeros 5

resultados = []

for mes in sorted(df_eval["mes_base"].unique()):

    df_mes = df_eval[df_eval["mes_base"] == mes].copy()
    df_mes = df_mes.sort_values("score", ascending=False).reset_index(drop=True)

    # Crear grupos 1..500 (ordenados por score)
    df_mes["grupo"] = (df_mes.index // (len(df_mes) / N_GROUPS)).astype(int) + 1
    df_mes.loc[df_mes["grupo"] > N_GROUPS, "grupo"] = N_GROUPS

    # Resumen por grupo
    tabla = df_mes.groupby("grupo").agg(
        count=("target", "size"),
        events=("target", "sum"),
        precision=("target", "mean"),
        score_mean=("score", "mean")
    ).reset_index()

    # Ordenados de mejor a peor
    tabla = tabla.sort_values("score_mean", ascending=False).reset_index(drop=True)

    # Recall acumulado
    total_eventos_mes = tabla["events"].sum()
    tabla["recall_acum"] = tabla["events"].cumsum() / total_eventos_mes

    # Lift
    tasa_base = df_mes["target"].mean()
    tabla["lift"] = tabla["precision"] / tasa_base

    # Mantener solo los TOP 5 del mes
    tabla = tabla.head(TOP_SHOW)
    tabla["mes_base"] = mes

    resultados.append(tabla)

# Unir todo
df_top_grupos = pd.concat(resultados, axis=0, ignore_index=True)

# Mostrar
print("\n📊 EFECTIVIDAD — TOP 5 DE 500 GRUPOS POR MES (Precision, Recall, Lift)")
print(df_top_grupos)


In [ ]:
#solo variables del modelo de Camila no estaN ['trx_q_abonos_6m_total', 'trx_q_abonos_6m_efectivo', 'trx_q_cargos_6m_total']

In [ ]:
camila = [
    'nivel_riesgo_lsb_ultima',
    'flag_desv_activa',
    'flag_ros_hist',
    'flag_casos_hist',
    'cp_cantidad_ing',
    'trx_monto_abonos_6m_total',
    'trx_monto_abonos_ratio_6m_efectivo_total',
    'q_meses_ingresos_0',
    'pasivo_soles',
    'edad_constitucion',
    'flag_variacion_abono_monto_total_5m_1m',
    'flag_variacion_efect_cargos_monto_5m_1m',
    'antiguedad',
    'dif_monto_abonos_cargos_efectivo_6m',
    'dif_q_abonos_cargos_efectivo_6m','mes_base', 'tipo_alerta_n2', 'num_documento','target_m'
]


In [ ]:
df_4=df_dataset[camila]

In [ ]:
import pandas as pd
from sklearn.utils import resample

# ============================================================
# 1️⃣ DEFINIR SPLIT DE DATOS
# ============================================================

df_train_raw = df_4[df_4["mes_base"] < 202506].copy()
df_val_raw   = df_4[df_4["mes_base"].isin([202506, 202507])].copy()
df_test_raw  = df_4[df_4["mes_base"].isin([202508, 202509])].copy()


# ============================================================
# 2️⃣ EN TRAIN: separar con alerta y sin alerta
# ============================================================

df_train_con_alerta  = df_train_raw[df_train_raw["tipo_alerta_n2"] != "0"]
df_train_sin_alerta  = df_train_raw[df_train_raw["tipo_alerta_n2"] == "0"]


# ============================================================
# 3️⃣ Separar clases en TRAIN
# ============================================================

df_train_1 = df_train_con_alerta[df_train_con_alerta["target_m"] == 1]
df_train_0 = df_train_con_alerta[df_train_con_alerta["target_m"] == 0]


# ============================================================
# 4️⃣ Balanceo:
#        Queremos 0.5% → target=1
#        Queremos 99.5% → target=0
# ============================================================

n_pos = len(df_train_1)
n_neg_deseado = int(n_pos * (99.5 / 0.5))  # relación 199:1

if n_neg_deseado <= len(df_train_0):
    df_train_0_bal = resample(df_train_0, replace=False, n_samples=n_neg_deseado, random_state=42)
else:
    df_train_0_bal = resample(df_train_0, replace=True, n_samples=n_neg_deseado, random_state=42)

df_train_1_bal = df_train_1


# ============================================================
# 5️⃣ Agregar 0.5% de casos SIN alerta
# ============================================================

df_sin_alerta_sample = df_train_sin_alerta.sample(frac=0.005, random_state=42)


# ============================================================
# 6️⃣ Construir TRAIN final
# ============================================================

df_train = pd.concat([
    df_train_0_bal,
    df_train_1_bal,
    df_sin_alerta_sample
], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)


# ============================================================
# 7️⃣ VALIDACIÓN → se deja intacta
# ============================================================

df_val = df_val_raw.copy()


# ============================================================
# 8️⃣ TEST → meses 202508 y 202509 completos
# ============================================================

df_test = df_test_raw.copy()


# ============================================================
# 9️⃣ SALIDAS
# ============================================================

print("📌 TRAIN (target):")
print(df_train["target_m"].value_counts(normalize=True))

print("\n📌 VAL (target):")
print(df_val["target_m"].value_counts(normalize=True))

print("\n📌 TEST (target):")
print(df_test["target_m"].value_counts(normalize=True))

print("\nTamaños finales:")
print("TRAIN:", df_train.shape)
print("VAL:  ", df_val.shape)
print("TEST: ", df_test.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# ============================================================
# 1️⃣ Separar TRAIN / VAL / TEST por mes_base
# ============================================================



# Guardamos para reconstruir después
df_all = pd.concat([df_train, df_val, df_test], axis=0).copy()

from sklearn.preprocessing import LabelEncoder
import pandas as pd

# ============================================================
# 1️⃣ Codificar categorías en df_all (NO hacemos splits aún)
# ============================================================

df_encoded = df_all.copy()

categoricas = [
    "flag_casos_hist"
]

encoders = {}

for c in categoricas:
    le = LabelEncoder()
    df_encoded[c] = le.fit_transform(df_encoded[c].astype(str))
    encoders[c] = le

# ============================================================
# 2️⃣ Ahora sí reconstruimos los splits con mes_base original
# ============================================================

df_train = df_encoded[df_encoded["mes_base"] <= 202505].copy()       # Train
df_val   = df_encoded[df_encoded["mes_base"].isin([202506, 202507])].copy()  # Val
df_test  = df_encoded[df_encoded["mes_base"].isin([202508, 202509])].copy()  # Test

print("Train:", df_train.shape)
print("Val:  ", df_val.shape)
print("Test: ", df_test.shape)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from scipy.stats import randint, uniform

# =====================================================
# 1️⃣ Separar features / target
# =====================================================

target = "target_m"

X_train = df_train.drop(columns=[target, "mes_base", "tipo_alerta_n2", "num_documento"], errors="ignore")
y_train = df_train[target]

X_val   = df_val.drop(columns=[target, "mes_base", "tipo_alerta_n2", "num_documento"], errors="ignore")
y_val   = df_val[target]

X_test  = df_test.drop(columns=[target, "mes_base", "tipo_alerta_n2", "num_documento"], errors="ignore")
y_test  = df_test[target]

# =====================================================
# 2️⃣ Todas las variables son numéricas (LabelEncoded)
# =====================================================

pipeline_xgb = Pipeline(steps=[
    ("xgb", XGBClassifier(
        eval_metric="logloss",
        tree_method="hist",
        random_state=42
    ))
])

# =====================================================
# 3️⃣ HPO
# =====================================================

param_dist = {
    "xgb__max_depth": randint(3, 8),
    "xgb__learning_rate": uniform(0.01, 0.15),
    "xgb__subsample": uniform(0.6, 0.4),
    "xgb__colsample_bytree": uniform(0.6, 0.4),
    "xgb__gamma": uniform(0, 5),
    "xgb__min_child_weight": randint(1, 10),
    "xgb__n_estimators": randint(200, 800)
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

search_xgb = RandomizedSearchCV(
    estimator=pipeline_xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring="precision",
    cv=cv,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

search_xgb.fit(X_train, y_train)

print("\n🏆 Mejores hiperparámetros:")
print(search_xgb.best_params_)

# =====================================================
# 4️⃣ Evaluación en TEST
# =====================================================

y_pred_prob = search_xgb.predict_proba(X_test)[:,1]

def ks_score(y_true, y_score):
    df = pd.DataFrame({"y": y_true, "score": y_score})
    df = df.sort_values("score", ascending=False)
    df["cum_event"] = (df["y"]==1).cumsum() / df["y"].sum()
    df["cum_nonevent"] = (df["y"]==0).cumsum() / (len(df) - df["y"].sum())
    return (df["cum_event"] - df["cum_nonevent"]).max()

auc  = roc_auc_score(y_test, y_pred_prob)
gini = 2*auc - 1
ks   = ks_score(y_test, y_pred_prob)

print("\n📊 MÉTRICAS EN TEST:")
print(f"AUC   = {auc:.4f}")
print(f"Gini  = {gini:.4f}")
print(f"KS    = {ks:.4f}")

In [ ]:
# ===============================================================
# 1️⃣ Agregar score
# ===============================================================
df_val["score"]  = search_xgb.predict_proba(X_val)[:,1]
df_test["score"] = search_xgb.predict_proba(X_test)[:,1]

df_eval = pd.concat([df_val, df_test], axis=0).reset_index(drop=True)

# ===============================================================
# 2️⃣ Construir 1000 grupos por mes y mostrar los 10 MEJORES
# ===============================================================

N_GRUPOS = 500
resultados = []

for mes in sorted(df_eval["mes_base"].unique()):

    df_mes = df_eval[df_eval["mes_base"] == mes].copy()

    # Crear grupos de score (1000)
    df_mes["grupo_1000"] = pd.qcut(
        df_mes["score"].rank(method="first"),
        q=N_GRUPOS,
        labels=False
    ) + 1

    tabla = df_mes.groupby("grupo_1000").agg(
        count=("target_m", "size"),
        events=("target_m", "sum"),
        score_mean=("score", "mean")
    ).reset_index()

    tabla["precision"] = tabla["events"] / tabla["count"]

    total_eventos_mes = df_mes["target_m"].sum()
    tabla["recall_acum"] = tabla["events"].cumsum() / total_eventos_mes

    tabla["mes_base"] = mes

    # ❗ AHORA SÍ: mejores 10 → score más ALTO
    tabla = tabla.sort_values("score_mean", ascending=False).head(10)

    resultados.append(tabla)

# ===============================================================
# 3️⃣ Resultado final
# ===============================================================

df_top10_1000grupos = pd.concat(resultados, ignore_index=True)
df_top10_1000grupos = df_top10_1000grupos.sort_values(["mes_base", "score_mean"], ascending=[True, False])

print(df_top10_1000grupos)

In [ ]:
import pandas as pd

# ===============================================================
# CONFIG: cantidad de grupos
# ===============================================================
N_GROUPS = 500      # 500 grupos iguales
TOP_SHOW = 5        # mostrar solo los primeros 5

resultados = []

for mes in sorted(df_eval["mes_base"].unique()):

    df_mes = df_eval[df_eval["mes_base"] == mes].copy()
    df_mes = df_mes.sort_values("score", ascending=False).reset_index(drop=True)

    # Crear grupos 1..500 (ordenados por score)
    df_mes["grupo"] = (df_mes.index // (len(df_mes) / N_GROUPS)).astype(int) + 1
    df_mes.loc[df_mes["grupo"] > N_GROUPS, "grupo"] = N_GROUPS

    # Resumen por grupo
    tabla = df_mes.groupby("grupo").agg(
        count=("target_m", "size"),
        events=("target_m", "sum"),
        precision=("target_m", "mean"),
        score_mean=("score", "mean")
    ).reset_index()

    # Ordenados de mejor a peor
    tabla = tabla.sort_values("score_mean", ascending=False).reset_index(drop=True)

    # Recall acumulado
    total_eventos_mes = tabla["events"].sum()
    tabla["recall_acum"] = tabla["events"].cumsum() / total_eventos_mes

    # Lift
    tasa_base = df_mes["target_m"].mean()
    tabla["lift"] = tabla["precision"] / tasa_base

    # Mantener solo los TOP 5 del mes
    tabla = tabla.head(TOP_SHOW)
    tabla["mes_base"] = mes

    resultados.append(tabla)

# Unir todo
df_top_grupos = pd.concat(resultados, axis=0, ignore_index=True)

# Mostrar
print("\n📊 EFECTIVIDAD — TOP 5 DE 500 GRUPOS POR MES (Precision, Recall, Lift)")
print(df_top_grupos)


In [ ]:
import pandas as pd
import numpy as np

df_val["score"]  = search_xgb.predict_proba(X_val)[:,1]
df_test["score"] = search_xgb.predict_proba(X_test)[:,1]

df_eval = pd.concat([df_val, df_test], axis=0).reset_index(drop=True)

print("Meses incluidos:", df_eval["mes_base"].unique())

# ======================================================================
# 1️⃣ Definir percentiles acumulativos
# ======================================================================
percentiles = [0.01, 0.05, 0.10, 0.50, 1.00]
labels = ["Top 1%", "Top 5%", "Top 10%", "Top 50%", "Total"]

resultados = []

# ======================================================================
# 2️⃣ Loop por mes
# ======================================================================
for mes in sorted(df_eval["mes_base"].unique()):
    
    df_mes = df_eval[df_eval["mes_base"] == mes].copy()
    
    # Ordenar descendentemente por score
    df_mes = df_mes.sort_values("score", ascending=False).reset_index(drop=True)
    
    # Ranking en percentil
    df_mes["percentile"] = (df_mes.index + 1) / len(df_mes)

    tablas = []
    
    for p, label in zip(percentiles, labels):
        df_cut = df_mes[df_mes["percentile"] <= p]

        tabla = {
            "grupo": label,
            "count": len(df_cut),
            "events": df_cut["target_m"].sum(),
            "precision": df_cut["target_m"].mean(),
            "recall": df_cut["target_m"].sum() / df_mes["target_m"].sum(),
            "score_mean": df_cut["score"].mean(),
            "mes_base": mes
        }
        tablas.append(tabla)
    
    resultados.extend(tablas)

# ======================================================================
# 3️⃣ Resultado final
# ======================================================================
df_result_final = pd.DataFrame(resultados)

print("\n📊 EFECTIVIDAD ACUMULADA POR GRUPOS (1%,5%,10%,50%,100%)")
print(df_result_final)
